In [1]:
# Keep repository-relative paths valid from notebook subfolders.
from pathlib import Path
import os

os.chdir(next(
    root for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "notebooks").is_dir() and (root / "requirements.txt").is_file()
))

# CELL 1 - IMPORTS

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

In [2]:
# CELL 2 - FINAL PIPELINE ARTIFACT PATHS

# =====================================
# Historical startup failure dataset
# =====================================

STARTUP_FAILURE_DATA_FILE = Path(
    "data/startup_failure/startup-failure-insights.json"
)


# =====================================
# IDX benchmark
# =====================================

IDX_BENCHMARK_OVERALL_FILE = Path(
    "data/idx_financial/benchmark/idx_financial_benchmark_overall.csv"
)

IDX_BENCHMARK_PERIOD_FILE = Path(
    "data/idx_financial/benchmark/idx_financial_benchmark_by_period.csv"
)

IDX_RATIO_READY_FILE = Path(
    "data/idx_financial/benchmark/idx_financial_ratio_benchmark_ready.csv"
)


# =====================================
# Large company analogue dataset
# =====================================

COMPANY_EMBEDDINGS_FILE = Path(
    "data/company_dataset/retrieval/software_it_candidate_embeddings.npy"
)

COMPANY_METADATA_FILE = Path(
    "data/company_dataset/retrieval/software_it_candidate_metadata.csv"
)


# =====================================
# Local embedding model
# =====================================

EMBEDDING_MODEL_PATH = Path(
    "models/multilingual-minilm"
)


artifact_paths = {
    "startup_failures":
        STARTUP_FAILURE_DATA_FILE,

    "idx_benchmark_overall":
        IDX_BENCHMARK_OVERALL_FILE,

    "idx_benchmark_period":
        IDX_BENCHMARK_PERIOD_FILE,

    "idx_ratio_ready":
        IDX_RATIO_READY_FILE,

    "company_embeddings":
        COMPANY_EMBEDDINGS_FILE,

    "company_metadata":
        COMPANY_METADATA_FILE,

    "embedding_model":
        EMBEDDING_MODEL_PATH,
}


print("FINAL PIPELINE ARTIFACTS")

for name, path in artifact_paths.items():

    print(
        f"{name:25s}",
        "OK" if path.exists() else "MISSING",
        "->",
        path
    )

FINAL PIPELINE ARTIFACTS
startup_failures          OK -> data\startup-failure-insights.json
idx_benchmark_overall     OK -> data\idx_financial_benchmark_overall.csv
idx_benchmark_period      OK -> data\idx_financial_benchmark_by_period.csv
idx_ratio_ready           OK -> data\idx_financial_ratio_benchmark_ready.csv
company_embeddings        OK -> data\company_dataset\software_it_candidate_embeddings.npy
company_metadata          OK -> data\company_dataset\software_it_candidate_metadata.csv
embedding_model           OK -> models\multilingual-minilm


In [3]:
# CELL 3 - LOAD FINAL PIPELINE ARTIFACTS

# =====================================
# Startup failure evidence
# =====================================

with open(
    STARTUP_FAILURE_DATA_FILE,
    "r",
    encoding="utf-8"
) as file:

    startup_failure_data = json.load(
        file
    )


# =====================================
# IDX benchmark
# =====================================

idx_benchmark_overall_df = pd.read_csv(
    IDX_BENCHMARK_OVERALL_FILE
)

idx_benchmark_period_df = pd.read_csv(
    IDX_BENCHMARK_PERIOD_FILE
)

idx_ratio_ready_df = pd.read_csv(
    IDX_RATIO_READY_FILE,
    low_memory=False
)


# =====================================
# Company analogue artifacts
# =====================================

company_embeddings = np.load(
    COMPANY_EMBEDDINGS_FILE
)

company_metadata_df = pd.read_csv(
    COMPANY_METADATA_FILE,
    low_memory=False
)


print("ARTIFACT LOAD COMPLETE")

ARTIFACT LOAD COMPLETE


In [4]:
# CELL 4 - AUDIT LOADED PIPELINE ARTIFACTS

print("STARTUP FAILURE DATA")
print(
    "Cases:",
    len(startup_failure_data)
)


print("\nIDX BENCHMARK")
print(
    "Overall rows:",
    len(idx_benchmark_overall_df)
)

print(
    "Period rows:",
    len(idx_benchmark_period_df)
)

print(
    "Ratio-ready rows:",
    len(idx_ratio_ready_df)
)


print("\nCOMPANY ANALOGUES")
print(
    "Embedding shape:",
    company_embeddings.shape
)

print(
    "Metadata rows:",
    len(company_metadata_df)
)

print(
    "Unique company IDs:",
    company_metadata_df[
        "id"
    ].nunique()
)


assert (
    company_embeddings.shape[0]
    ==
    len(company_metadata_df)
)


assert (
    company_metadata_df[
        "id"
    ].nunique()
    ==
    len(company_metadata_df)
)


print(
    "\nCompany embedding alignment: OK"
)


print("\nIDX OVERALL BENCHMARK")

display(
    idx_benchmark_overall_df
)

STARTUP FAILURE DATA
Cases: 143

IDX BENCHMARK
Overall rows: 4
Period rows: 84
Ratio-ready rows: 14753

COMPANY ANALOGUES
Embedding shape: (115798, 384)
Metadata rows: 115798
Unique company IDs: 115798

Company embedding alignment: OK

IDX OVERALL BENCHMARK


,benchmark_scope,ratio,count,p05,p25,median,p75,p95
0,IDX_ALL,gross_margin,12332,-0.008175,0.125768,0.251172,0.437408,0.701815
1,IDX_ALL,debt_to_assets,14746,0.069907,0.255347,0.450176,0.660683,1.019874
2,IDX_ALL,cash_to_assets,13758,0.001782,0.015094,0.052363,0.133602,0.327047
3,IDX_ALL,ocf_to_revenue,12910,-0.891228,-0.044744,0.061471,0.199658,0.558679


In [5]:
# CELL 5 - LOAD EMBEDDING MODEL

embedding_model = SentenceTransformer(
    str(
        EMBEDDING_MODEL_PATH
    )
)


print(
    "Embedding model loaded:"
)

print(
    EMBEDDING_MODEL_PATH
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2972.95it/s]


Embedding model loaded:
models\multilingual-minilm


In [6]:
# CELL 6 - DEFINE COMPANY SEMANTIC SEARCH

def search_company_analogues(
    query,
    *,
    top_k=10,
):
    """
    Semantic search across precomputed company embeddings.
    """

    if not isinstance(query, str):
        query = str(query)

    query = query.strip()

    if not query:
        return pd.DataFrame()


    # ----------------------------------
    # Embed query only
    # ----------------------------------

    query_embedding = (
        embedding_model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True,
        )[0]
    )


    # ----------------------------------
    # Cosine similarity
    # Company embeddings are normalized
    # ----------------------------------

    similarity_scores = (
        company_embeddings
        @ query_embedding
    )


    top_k = min(
        int(top_k),
        len(similarity_scores)
    )


    if top_k <= 0:
        return pd.DataFrame()


    # ----------------------------------
    # Efficient top-k
    # ----------------------------------

    top_indices = np.argpartition(
        similarity_scores,
        -top_k
    )[
        -top_k:
    ]


    top_indices = top_indices[
        np.argsort(
            similarity_scores[
                top_indices
            ]
        )[::-1]
    ]


    results = (
        company_metadata_df
        .iloc[
            top_indices
        ]
        .copy()
    )


    results[
        "semantic_score"
    ] = similarity_scores[
        top_indices
    ]


    return (
        results
        .reset_index(drop=True)
    )

In [7]:
# CELL 7 - FORMAT COMPANY ANALOGUE RESULTS

employee_range_text_mapping = {
    "c_00001_00010":
        "1-10 employees",

    "c_00011_00050":
        "11-50 employees",

    "c_00051_00100":
        "51-100 employees",

    "c_00101_00250":
        "101-250 employees",

    "c_00251_00500":
        "251-500 employees",

    "c_00501_01000":
        "501-1000 employees",

    "c_01001_05000":
        "1001-5000 employees",

    "c_05001_10000":
        "5001-10000 employees",

    "c_10001_max":
        "10001+ employees",
}


def format_company_analogues(
    results_df,
    *,
    top_k=10,
):

    formatted_results = []


    for rank, (_, row) in enumerate(
        results_df
        .head(top_k)
        .iterrows(),
        start=1
    ):

        employee_code = (
            None
            if pd.isna(
                row.get(
                    "num_employees_enum"
                )
            )
            else str(
                row[
                    "num_employees_enum"
                ]
            )
        )


        founded_year = (
            None
            if pd.isna(
                row.get(
                    "founded_year"
                )
            )
            else int(
                float(
                    row[
                        "founded_year"
                    ]
                )
            )
        )


        formatted_results.append(
            {
                "rank":
                    rank,

                "id":
                    None
                    if pd.isna(
                        row.get("id")
                    )
                    else str(
                        row["id"]
                    ),

                "name":
                    None
                    if pd.isna(
                        row.get("name")
                    )
                    else str(
                        row["name"]
                    ),

                "semantic_score":
                    round(
                        float(
                            row[
                                "semantic_score"
                            ]
                        ),
                        6
                    ),

                "short_description":
                    None
                    if pd.isna(
                        row.get(
                            "short_description"
                        )
                    )
                    else str(
                        row[
                            "short_description"
                        ]
                    ),

                "categories":
                    None
                    if pd.isna(
                        row.get(
                            "categories"
                        )
                    )
                    else str(
                        row[
                            "categories"
                        ]
                    ),

                "locations":
                    None
                    if pd.isna(
                        row.get(
                            "locations"
                        )
                    )
                    else str(
                        row[
                            "locations"
                        ]
                    ),

                "employee_range":
                    None
                    if employee_code is None
                    else employee_range_text_mapping.get(
                        employee_code,
                        employee_code
                    ),

                "founded_year":
                    founded_year,
            }
        )


    return formatted_results

In [8]:
# CELL 8 - TEST COMPANY ANALOGUE SEARCH

test_company_query = """
A small early-stage AI SaaS company that helps
businesses automate repetitive operational workflows.
""".strip()


start_time = time.perf_counter()


company_search_df = (
    search_company_analogues(
        test_company_query,
        top_k=10
    )
)


formatted_company_results = (
    format_company_analogues(
        company_search_df,
        top_k=10
    )
)


elapsed_ms = (
    time.perf_counter()
    - start_time
) * 1000


print(
    "Search time:",
    round(
        elapsed_ms,
        2
    ),
    "ms"
)

print(
    "Results:",
    len(
        formatted_company_results
    )
)


for result in formatted_company_results:

    print(
        "\n" + "=" * 80
    )

    print(
        f"#{result['rank']} "
        f"{result['name']}"
    )

    print(
        "Score:",
        result[
            "semantic_score"
        ]
    )

    print(
        "Description:",
        result[
            "short_description"
        ]
    )

    print(
        "Categories:",
        result[
            "categories"
        ]
    )

Search time: 109.37 ms
Results: 10

#1 Aiver.ai
Score: 0.746031
Description: Unlock The Power Of Automation. Automate workflows with AIVER using the power of AI and ML to make the workflows smarter over time.
Categories: Accounting, Artificial Intelligence (AI), Business Intelligence, Compliance, Data Visualization, Fraud Detection, Information Technology, Machine Learning, Predictive Analytics, Product Management

#2 Automations.io
Score: 0.726555
Description: Automations.io specializes in a no-code automation platform to visually automate repetitive work and build workflows to save time.
Categories: Marketing Automation, Sales Automation, Software

#3 Businessflow AI
Score: 0.723348
Description: AI powered SaaS for the next generation recruitment.
Categories: Artificial Intelligence (AI), Business Information Systems, Business Process Automation (BPA), Enterprise Software, Human Resources, Recruiting, Software

#4 Handy.ai
Score: 0.716529
Description: Handy.ai combines process automa

In [9]:
# CELL 9 - INSPECT STARTUP FAILURE DATA STRUCTURE

print(
    "Startup failure cases:",
    len(startup_failure_data)
)

print(
    "\nTop-level type:",
    type(startup_failure_data)
)


if len(startup_failure_data) > 0:

    first_case = (
        startup_failure_data[0]
    )

    print(
        "\nFirst case keys:"
    )

    print(
        list(
            first_case.keys()
        )
    )


    print(
        "\nFIRST CASE SAMPLE"
    )

    print(
        json.dumps(
            first_case,
            indent=2,
            ensure_ascii=False
        )[:5000]
    )

Startup failure cases: 143

Top-level type: <class 'list'>

First case keys:
['company', 'funding', 'category', 'investors', 'failure_reason', 'prompt', 'completion']

FIRST CASE SAMPLE
{
  "company": "AdBrite",
  "funding": "$35M",
  "category": "Total funding from $25M - $50M",
  "investors": [
    "Sequoia Capital",
    "Artis Capital Management"
  ],
  "failure_reason": "Despite claiming to be the largest independent ad exchange and at one time being seen as a serious competitor to Google Adwords, it seems that they were unable to make enough money or sell the company to potential buyers.",
  "prompt": "Why couldn't AdBrite compete with Google AdWords?",
  "completion": "AdBrite failed to compete with Google AdWords despite advantages: 1) Although they claimed to be the largest independent ad exchange, they couldn't generate sufficient revenue, 2) Despite being seen as a serious competitor to Google AdWords, they couldn't achieve profitability, 3) No potential buyers were willing t

In [12]:
# CELL 10 - BUILD STARTUP FAILURE RETRIEVAL DOCUMENTS
# REVISED TO MATCH ACTUAL JSON SCHEMA

def get_failure_case_text(
    case
):
    """
    Build deterministic retrieval text from
    historical startup failure evidence.

    Important:
    - completion is intentionally excluded
    - failure_reason is the primary evidence
    """

    parts = []


    company = case.get(
        "company"
    )

    if company:
        parts.append(
            f"Company: {str(company).strip()}"
        )


    category = case.get(
        "category"
    )

    if category:
        parts.append(
            f"Category: {str(category).strip()}"
        )


    funding = case.get(
        "funding"
    )

    if funding:
        parts.append(
            f"Funding: {str(funding).strip()}"
        )


    failure_reason = case.get(
        "failure_reason"
    )

    if failure_reason:
        parts.append(
            "Failure Reason: "
            + str(
                failure_reason
            ).strip()
        )


    prompt = case.get(
        "prompt"
    )

    if prompt:
        parts.append(
            f"Failure Question: {str(prompt).strip()}"
        )


    return "\n".join(
        parts
    )


startup_failure_documents = []


for index, case in enumerate(
    startup_failure_data
):

    retrieval_text = (
        get_failure_case_text(
            case
        )
    )


    startup_failure_documents.append(
        {
            "case_index":
                index,

            "retrieval_text":
                retrieval_text,

            "raw_case":
                case,
        }
    )


print(
    "Failure documents:",
    len(
        startup_failure_documents
    )
)


print(
    "Empty retrieval documents:",
    sum(
        not item[
            "retrieval_text"
        ].strip()
        for item
        in startup_failure_documents
    )
)


print(
    "\nSAMPLE RETRIEVAL TEXT"
)

print(
    startup_failure_documents[0][
        "retrieval_text"
    ]
)

Failure documents: 143
Empty retrieval documents: 0

SAMPLE RETRIEVAL TEXT
Company: AdBrite
Category: Total funding from $25M - $50M
Funding: $35M
Failure Reason: Despite claiming to be the largest independent ad exchange and at one time being seen as a serious competitor to Google Adwords, it seems that they were unable to make enough money or sell the company to potential buyers.
Failure Question: Why couldn't AdBrite compete with Google AdWords?


In [13]:
# CELL 11 - EMBED STARTUP FAILURE DOCUMENTS

failure_texts = [
    item[
        "retrieval_text"
    ]
    for item
    in startup_failure_documents
]


start_time = (
    time.perf_counter()
)


startup_failure_embeddings = (
    embedding_model.encode(
        failure_texts,

        batch_size=32,

        show_progress_bar=True,

        convert_to_numpy=True,

        normalize_embeddings=True,
    )
)


elapsed_seconds = (
    time.perf_counter()
    - start_time
)


print(
    "\nSTARTUP FAILURE EMBEDDING COMPLETE"
)

print(
    "Shape:",
    startup_failure_embeddings.shape
)

print(
    "Elapsed seconds:",
    round(
        elapsed_seconds,
        2
    )
)

Batches: 100%|██████████| 5/5 [00:03<00:00,  1.43it/s]


STARTUP FAILURE EMBEDDING COMPLETE
Shape: (143, 384)
Elapsed seconds: 3.5


In [14]:
# CELL 12 - DEFINE HISTORICAL FAILURE SEMANTIC SEARCH

def search_historical_failures(
    query,
    *,
    top_k=5,
):
    """
    Semantic search across historical startup
    failure cases.
    """

    if not isinstance(
        query,
        str
    ):
        query = str(
            query
        )


    query = query.strip()


    if not query:
        return []


    query_embedding = (
        embedding_model.encode(
            [query],

            convert_to_numpy=True,

            normalize_embeddings=True,
        )[0]
    )


    similarity_scores = (
        startup_failure_embeddings
        @ query_embedding
    )


    top_k = min(
        int(top_k),
        len(
            similarity_scores
        )
    )


    if top_k <= 0:
        return []


    top_indices = np.argpartition(
        similarity_scores,
        -top_k
    )[
        -top_k:
    ]


    top_indices = top_indices[
        np.argsort(
            similarity_scores[
                top_indices
            ]
        )[::-1]
    ]


    results = []


    for rank, index in enumerate(
        top_indices,
        start=1
    ):

        document = (
            startup_failure_documents[
                int(index)
            ]
        )

        case = document[
            "raw_case"
        ]


        results.append(
            {
                "rank":
                    rank,

                "case_index":
                    int(
                        index
                    ),

                "company":
                    case.get(
                        "company"
                    ),

                "semantic_score":
                    round(
                        float(
                            similarity_scores[
                                index
                            ]
                        ),
                        6
                    ),

                "funding":
                    case.get(
                        "funding"
                    ),

                "category":
                    case.get(
                        "category"
                    ),

                "investors":
                    case.get(
                        "investors"
                    ),

                "failure_reason":
                    case.get(
                        "failure_reason"
                    ),

                "prompt":
                    case.get(
                        "prompt"
                    ),
            }
        )


    return results

In [15]:
# CELL 13 - TEST HISTORICAL FAILURE SEARCH

test_failure_query = """
A small AI SaaS startup assumes that strong product
technology and rapid growth will be enough to reach
sustainable profitability despite intense competition.
""".strip()


failure_search_results = (
    search_historical_failures(
        test_failure_query,
        top_k=5
    )
)


print(
    "Results:",
    len(
        failure_search_results
    )
)


for result in failure_search_results:

    print(
        "\n" + "=" * 90
    )

    print(
        f"#{result['rank']} "
        f"{result['company']}"
    )

    print(
        "Score:",
        result[
            "semantic_score"
        ]
    )

    print(
        "Failure Reason:",
        result[
            "failure_reason"
        ]
    )

Results: 5

#1 Argo AI
Score: 0.388682
Failure Reason: Ford said in its third-quarter earnings report that it made a strategic decision to shift its resources to developing advanced driver assistance systems, and not autonomous vehicle technology that can be applied to robotaxis. That decision appears to have been fueled by Argo's inability to attract new investors. Ford CEO Jim Farley acknowledged that the company anticipated being able to bring autonomous vehicle technology broadly to market by 2021.

#2 Lighthouse AI
Score: 0.381145
Failure Reason: Lighthouse AI, a smarthome startup, stated on its website: To our customers, investors, family, friends, partners, fans, and everyone else involved in the Lighthouse journey: It's been a pleasure, and we can't thank you enough for your support along the way. I am incredibly proud of the groundbreaking work the Lighthouse team accomplished – delivering useful and accessible intelligence for our homes via advanced AI and 3D sensing. Unfortu

In [16]:
# CELL 14 - RETRIEVE HISTORICAL FAILURES PER ASSUMPTION

def retrieve_failure_evidence_for_assumptions(
    assumptions,
    *,
    top_k_per_assumption=5,
):
    """
    Retrieve historical failure evidence
    for multiple assumptions.
    """

    outputs = []


    for index, item in enumerate(
        assumptions,
        start=1
    ):

        if isinstance(
            item,
            dict
        ):

            assumption_id = (
                item.get("id")
                or item.get(
                    "assumption_id"
                )
                or f"A{index}"
            )


            assumption_text = (
                item.get(
                    "retrieval_query"
                )
                or item.get(
                    "semantic_query"
                )
                or item.get(
                    "assumption"
                )
                or item.get(
                    "text"
                )
                or ""
            )


            original_assumption = (
                item.get(
                    "assumption"
                )
                or assumption_text
            )


        else:

            assumption_id = (
                f"A{index}"
            )

            assumption_text = str(
                item
            )

            original_assumption = (
                assumption_text
            )


        assumption_text = (
            str(
                assumption_text
            ).strip()
        )


        if not assumption_text:
            continue


        failure_results = (
            search_historical_failures(
                assumption_text,
                top_k=
                    top_k_per_assumption
            )
        )


        outputs.append(
            {
                "assumption_id":
                    str(
                        assumption_id
                    ),

                "assumption":
                    str(
                        original_assumption
                    ),

                "retrieval_query":
                    assumption_text,

                "historical_failure_count":
                    len(
                        failure_results
                    ),

                "historical_failures":
                    failure_results,
            }
        )


    return outputs

In [17]:
# CELL 15 - NORMALIZE ASSUMPTIONS FOR INTEGRATED EVIDENCE

def normalize_assumptions(
    assumptions
):
    normalized = []

    for index, item in enumerate(
        assumptions,
        start=1
    ):

        if isinstance(item, dict):

            assumption_id = (
                item.get("id")
                or item.get("assumption_id")
                or f"A{index}"
            )

            assumption_text = (
                item.get("assumption")
                or item.get("text")
                or item.get("description")
                or ""
            )

            retrieval_query = (
                item.get("retrieval_query")
                or item.get("semantic_query")
                or item.get("search_query")
                or assumption_text
            )

        else:

            assumption_id = f"A{index}"
            assumption_text = str(item)
            retrieval_query = assumption_text


        assumption_text = str(
            assumption_text
        ).strip()

        retrieval_query = str(
            retrieval_query
        ).strip()


        if not assumption_text:
            continue


        if not retrieval_query:
            retrieval_query = (
                assumption_text
            )


        normalized.append(
            {
                "id":
                    str(assumption_id),

                "assumption":
                    assumption_text,

                "retrieval_query":
                    retrieval_query,
            }
        )


    return normalized

In [18]:
# CELL 16 - RETRIEVE INTEGRATED QUALITATIVE EVIDENCE

def retrieve_integrated_evidence(
    assumptions,
    *,
    top_company_analogues=5,
    top_failure_cases=5,
):
    normalized_assumptions = (
        normalize_assumptions(
            assumptions
        )
    )

    outputs = []


    for item in normalized_assumptions:

        query = (
            item["retrieval_query"]
        )


        # ==================================
        # Company analogue evidence
        # ==================================

        company_search_df = (
            search_company_analogues(
                query,
                top_k=
                    top_company_analogues
            )
        )


        company_results = (
            format_company_analogues(
                company_search_df,
                top_k=
                    top_company_analogues
            )
        )


        # ==================================
        # Historical failure evidence
        # ==================================

        failure_results = (
            search_historical_failures(
                query,
                top_k=
                    top_failure_cases
            )
        )


        outputs.append(
            {
                "assumption_id":
                    item["id"],

                "assumption":
                    item["assumption"],

                "retrieval_query":
                    query,

                "company_analogue_count":
                    len(
                        company_results
                    ),

                "company_analogues":
                    company_results,

                "historical_failure_count":
                    len(
                        failure_results
                    ),

                "historical_failures":
                    failure_results,
            }
        )


    return {
        "assumption_count":
            len(outputs),

        "company_candidate_universe_size":
            len(
                company_metadata_df
            ),

        "historical_failure_universe_size":
            len(
                startup_failure_documents
            ),

        "assumptions":
            outputs,
    }

In [19]:
# CELL 17 - TEST INTEGRATED QUALITATIVE EVIDENCE

test_assumptions = [
    {
        "id": "A1",

        "assumption": (
            "Small businesses are willing to pay "
            "for AI-powered workflow automation software."
        ),

        "retrieval_query": (
            "Small AI SaaS companies providing "
            "workflow automation software "
            "for business customers."
        ),
    },

    {
        "id": "A2",

        "assumption": (
            "A small software team can build "
            "and operate a scalable SaaS platform."
        ),

        "retrieval_query": (
            "Small SaaS software companies "
            "building scalable business software "
            "with limited teams."
        ),
    },
]


integrated_evidence = (
    retrieve_integrated_evidence(
        test_assumptions,
        top_company_analogues=5,
        top_failure_cases=5,
    )
)


print(
    "Assumptions:",
    integrated_evidence[
        "assumption_count"
    ]
)

print(
    "Company universe:",
    integrated_evidence[
        "company_candidate_universe_size"
    ]
)

print(
    "Failure universe:",
    integrated_evidence[
        "historical_failure_universe_size"
    ]
)


for item in integrated_evidence[
    "assumptions"
]:

    print(
        "\n" + "=" * 100
    )

    print(
        "Assumption:",
        item["assumption"]
    )


    print(
        "\nTOP COMPANY ANALOGUES"
    )

    for company in item[
        "company_analogues"
    ]:

        print(
            f"  #{company['rank']} "
            f"{company['name']} "
            f"({company['semantic_score']})"
        )


    print(
        "\nTOP HISTORICAL FAILURES"
    )

    for failure in item[
        "historical_failures"
    ]:

        print(
            f"  #{failure['rank']} "
            f"{failure['company']} "
            f"({failure['semantic_score']})"
        )

Assumptions: 2
Company universe: 115798
Failure universe: 143

Assumption: Small businesses are willing to pay for AI-powered workflow automation software.

TOP COMPANY ANALOGUES
  #1 Businessflow AI (0.733688)
  #2 Aiver.ai (0.730851)
  #3 Handy.ai (0.72375)
  #4 Work Simplr (0.718333)
  #5 Workorder AI (0.715348)

TOP HISTORICAL FAILURES
  #1 Argo AI (0.388141)
  #2 Lighthouse AI (0.37005)
  #3 Pearl Automation (0.360892)
  #4 Jumpshot (0.314447)
  #5 IRL (0.31408)

Assumption: A small software team can build and operate a scalable SaaS platform.

TOP COMPANY ANALOGUES
  #1 TinySeed (0.687822)
  #2 Small Software (0.685253)
  #3 Little SaaS, (0.683633)
  #4 1inch Limited (0.677163)
  #5 Rolocost Interactive (0.675483)

TOP HISTORICAL FAILURES
  #1 ScaleFactor (0.370675)
  #2 Shyp (0.34715)
  #3 Brodmann17 (0.340397)
  #4 Oryx Vision (0.338598)
  #5 Automatic (0.334354)


In [20]:
# CELL 18 - DEFINE DETERMINISTIC FINANCIAL STRESS TEST

def run_financial_stress_test(
    *,
    monthly_orders,
    revenue_per_order,
    variable_cost_per_order,
    promo_subsidy_per_order=0,
    delivery_cost_per_order=0,
    monthly_fixed_cost=0,
    monthly_driver_cost=0,
    cash_balance=None,
):
    """
    Deterministic unit economics and runway calculation.

    Missing values are NOT interpreted as zero,
    except optional cost inputs whose defaults are
    explicitly zero.
    """

    monthly_orders = float(monthly_orders)
    revenue_per_order = float(revenue_per_order)
    variable_cost_per_order = float(
        variable_cost_per_order
    )

    promo_subsidy_per_order = float(
        promo_subsidy_per_order
    )

    delivery_cost_per_order = float(
        delivery_cost_per_order
    )

    monthly_fixed_cost = float(
        monthly_fixed_cost
    )

    monthly_driver_cost = float(
        monthly_driver_cost
    )


    # -----------------------------------
    # Per-order economics
    # -----------------------------------

    contribution_per_order = (
        revenue_per_order
        - variable_cost_per_order
        - promo_subsidy_per_order
        - delivery_cost_per_order
    )


    contribution_margin = (
        contribution_per_order
        / revenue_per_order
        if revenue_per_order != 0
        else np.nan
    )


    # -----------------------------------
    # Monthly economics
    # -----------------------------------

    monthly_revenue = (
        monthly_orders
        * revenue_per_order
    )


    monthly_variable_cost = (
        monthly_orders
        * variable_cost_per_order
    )


    monthly_promo_subsidy = (
        monthly_orders
        * promo_subsidy_per_order
    )


    monthly_delivery_cost = (
        monthly_orders
        * delivery_cost_per_order
    )


    monthly_contribution = (
        monthly_orders
        * contribution_per_order
    )


    monthly_operating_cost = (
        monthly_fixed_cost
        + monthly_driver_cost
    )


    monthly_operating_profit = (
        monthly_contribution
        - monthly_operating_cost
    )


    monthly_burn = max(
        -monthly_operating_profit,
        0
    )


    # -----------------------------------
    # Runway
    # -----------------------------------

    if cash_balance is None:
        runway_months = None

    else:
        cash_balance = float(
            cash_balance
        )

        if monthly_burn > 0:
            runway_months = (
                cash_balance
                / monthly_burn
            )
        else:
            runway_months = None


    # -----------------------------------
    # Break-even volume
    # -----------------------------------

    if contribution_per_order > 0:

        break_even_orders = (
            monthly_operating_cost
            / contribution_per_order
        )

    else:
        break_even_orders = None


    return {
        "monthly_orders":
            monthly_orders,

        "revenue_per_order":
            revenue_per_order,

        "variable_cost_per_order":
            variable_cost_per_order,

        "promo_subsidy_per_order":
            promo_subsidy_per_order,

        "delivery_cost_per_order":
            delivery_cost_per_order,

        "contribution_per_order":
            contribution_per_order,

        "contribution_margin":
            contribution_margin,

        "monthly_revenue":
            monthly_revenue,

        "monthly_variable_cost":
            monthly_variable_cost,

        "monthly_promo_subsidy":
            monthly_promo_subsidy,

        "monthly_delivery_cost":
            monthly_delivery_cost,

        "monthly_contribution":
            monthly_contribution,

        "monthly_fixed_cost":
            monthly_fixed_cost,

        "monthly_driver_cost":
            monthly_driver_cost,

        "monthly_operating_profit":
            monthly_operating_profit,

        "monthly_burn":
            monthly_burn,

        "runway_months":
            runway_months,

        "break_even_orders":
            break_even_orders,
    }

In [21]:
# CELL 19 - BUILD DIRECTIONAL IDX BENCHMARK COMPARISON

def benchmark_against_idx(
    financial_result,
    idx_benchmark_df,
):
    """
    Compare available startup/unit-economics metrics
    against IDX benchmark directionally.

    Important:
    contribution_margin != accounting gross_margin.
    This comparison is directional only.
    """

    comparisons = []


    contribution_margin = (
        financial_result.get(
            "contribution_margin"
        )
    )


    if (
        contribution_margin is not None
        and not pd.isna(
            contribution_margin
        )
    ):

        benchmark_row = (
            idx_benchmark_df[
                idx_benchmark_df[
                    "ratio"
                ]
                .eq(
                    "gross_margin"
                )
            ]
        )


        if len(benchmark_row) == 1:

            row = (
                benchmark_row
                .iloc[0]
            )


            value = float(
                contribution_margin
            )

            p25 = float(
                row["p25"]
            )

            median = float(
                row["median"]
            )

            p75 = float(
                row["p75"]
            )


            if value < p25:
                position = "below_p25"

            elif value < median:
                position = "between_p25_and_median"

            elif value < p75:
                position = "between_median_and_p75"

            else:
                position = "above_p75"


            comparisons.append(
                {
                    "startup_metric":
                        "contribution_margin",

                    "benchmark_metric":
                        "gross_margin",

                    "comparison_type":
                        "directional_only",

                    "startup_value":
                        value,

                    "idx_p25":
                        p25,

                    "idx_median":
                        median,

                    "idx_p75":
                        p75,

                    "position":
                        position,

                    "note": (
                        "Contribution margin is not "
                        "identical to accounting gross "
                        "margin. IDX comparison is "
                        "directional only."
                    ),
                }
            )


    return comparisons

In [22]:
# CELL 20 - BUILD FINANCIAL EVIDENCE PAYLOAD

def build_financial_evidence(
    financial_inputs
):
    """
    Run deterministic stress test and attach
    directional IDX benchmark evidence.
    """

    stress_test = (
        run_financial_stress_test(
            **financial_inputs
        )
    )


    idx_comparison = (
        benchmark_against_idx(
            stress_test,
            idx_benchmark_overall_df
        )
    )


    flags = []


    contribution_margin = (
        stress_test[
            "contribution_margin"
        ]
    )


    operating_profit = (
        stress_test[
            "monthly_operating_profit"
        ]
    )


    runway_months = (
        stress_test[
            "runway_months"
        ]
    )


    if (
        not pd.isna(
            contribution_margin
        )
        and contribution_margin <= 0
    ):

        flags.append(
            {
                "code":
                    "NON_POSITIVE_CONTRIBUTION_MARGIN",

                "severity":
                    "high",

                "message":
                    (
                        "Contribution margin is "
                        "non-positive."
                    ),
            }
        )


    if operating_profit < 0:

        flags.append(
            {
                "code":
                    "NEGATIVE_OPERATING_PROFIT",

                "severity":
                    "high",

                "message":
                    (
                        "Monthly operating profit "
                        "is negative under the "
                        "provided assumptions."
                    ),
            }
        )


    if (
        runway_months is not None
        and runway_months < 6
    ):

        flags.append(
            {
                "code":
                    "SHORT_RUNWAY",

                "severity":
                    "high",

                "message":
                    (
                        "Estimated runway is below "
                        "6 months."
                    ),
            }
        )


    return {
        "stress_test":
            stress_test,

        "idx_benchmark":
            idx_comparison,

        "flags":
            flags,
    }

In [23]:
# CELL 21 - TEST FINANCIAL EVIDENCE

test_financial_inputs = {
    "monthly_orders":
        2_000,

    "revenue_per_order":
        100_000,

    "variable_cost_per_order":
        45_000,

    "promo_subsidy_per_order":
        10_000,

    "delivery_cost_per_order":
        15_000,

    "monthly_fixed_cost":
        80_000_000,

    "monthly_driver_cost":
        10_000_000,

    "cash_balance":
        500_000_000,
}


financial_evidence = (
    build_financial_evidence(
        test_financial_inputs
    )
)


print(
    json.dumps(
        financial_evidence,
        indent=2,
        ensure_ascii=False
    )
)

{
  "stress_test": {
    "monthly_orders": 2000.0,
    "revenue_per_order": 100000.0,
    "variable_cost_per_order": 45000.0,
    "promo_subsidy_per_order": 10000.0,
    "delivery_cost_per_order": 15000.0,
    "contribution_per_order": 30000.0,
    "contribution_margin": 0.3,
    "monthly_revenue": 200000000.0,
    "monthly_variable_cost": 90000000.0,
    "monthly_promo_subsidy": 20000000.0,
    "monthly_delivery_cost": 30000000.0,
    "monthly_contribution": 60000000.0,
    "monthly_fixed_cost": 80000000.0,
    "monthly_driver_cost": 10000000.0,
    "monthly_operating_profit": -30000000.0,
    "monthly_burn": 30000000.0,
    "runway_months": 16.666666666666668,
    "break_even_orders": 3000.0
  },
  "idx_benchmark": [
    {
      "startup_metric": "contribution_margin",
      "benchmark_metric": "gross_margin",
      "comparison_type": "directional_only",
      "startup_value": 0.3,
      "idx_p25": 0.1257684664467831,
      "idx_median": 0.2511718666577408,
      "idx_p75": 0.4374083

In [24]:
# CELL 22 - BUILD INTEGRATED ANALYSIS EVIDENCE

def build_integrated_analysis_evidence(
    assumptions,
    *,
    financial_inputs=None,
    top_company_analogues=5,
    top_failure_cases=5,
):
    """
    Combine qualitative retrieval evidence
    and deterministic financial evidence.
    """

    # ==================================
    # Qualitative evidence
    # ==================================

    qualitative_evidence = (
        retrieve_integrated_evidence(
            assumptions,
            top_company_analogues=
                top_company_analogues,
            top_failure_cases=
                top_failure_cases,
        )
    )


    # ==================================
    # Financial evidence
    # ==================================

    if financial_inputs is not None:

        financial_evidence = (
            build_financial_evidence(
                financial_inputs
            )
        )

    else:

        financial_evidence = None


    return {
        "qualitative_evidence":
            qualitative_evidence,

        "financial_evidence":
            financial_evidence,
    }

In [25]:
# CELL 23 - BUILD ANALYSIS SUMMARY METRICS

def build_analysis_summary(
    integrated_analysis
):
    qualitative = (
        integrated_analysis[
            "qualitative_evidence"
        ]
    )

    financial = (
        integrated_analysis[
            "financial_evidence"
        ]
    )


    summary = {
        "assumption_count":
            qualitative[
                "assumption_count"
            ],

        "company_candidate_universe_size":
            qualitative[
                "company_candidate_universe_size"
            ],

        "historical_failure_universe_size":
            qualitative[
                "historical_failure_universe_size"
            ],
    }


    if financial is not None:

        stress_test = (
            financial[
                "stress_test"
            ]
        )

        summary.update(
            {
                "contribution_margin":
                    stress_test[
                        "contribution_margin"
                    ],

                "monthly_operating_profit":
                    stress_test[
                        "monthly_operating_profit"
                    ],

                "monthly_burn":
                    stress_test[
                        "monthly_burn"
                    ],

                "runway_months":
                    stress_test[
                        "runway_months"
                    ],

                "break_even_orders":
                    stress_test[
                        "break_even_orders"
                    ],

                "financial_flag_count":
                    len(
                        financial[
                            "flags"
                        ]
                    ),
            }
        )


    return summary

In [26]:
# CELL 24 - TEST INTEGRATED ANALYSIS PAYLOAD

integrated_analysis = (
    build_integrated_analysis_evidence(
        test_assumptions,

        financial_inputs=
            test_financial_inputs,

        top_company_analogues=5,

        top_failure_cases=5,
    )
)


analysis_summary = (
    build_analysis_summary(
        integrated_analysis
    )
)


print(
    "ANALYSIS SUMMARY"
)

print(
    json.dumps(
        analysis_summary,
        indent=2,
        ensure_ascii=False
    )
)


print(
    "\nTOP-LEVEL PAYLOAD KEYS"
)

print(
    integrated_analysis.keys()
)

ANALYSIS SUMMARY
{
  "assumption_count": 2,
  "company_candidate_universe_size": 115798,
  "historical_failure_universe_size": 143,
  "contribution_margin": 0.3,
  "monthly_operating_profit": -30000000.0,
  "monthly_burn": 30000000.0,
  "runway_months": 16.666666666666668,
  "break_even_orders": 3000.0,
  "financial_flag_count": 1
}

TOP-LEVEL PAYLOAD KEYS
dict_keys(['qualitative_evidence', 'financial_evidence'])


In [27]:
# CELL 25 - DEFINE FINAL ANALYZE_DECISION WRAPPER

def analyze_decision(
    *,
    decision,
    assumptions,
    financial_inputs=None,
    top_company_analogues=5,
    top_failure_cases=5,
):
    """
    Final deterministic evidence orchestration.

    Note:
    - Assumptions are expected to come from the
      assumption analyzer.
    - This function does not invent assumptions.
    - Financial calculations remain deterministic.
    """

    decision = str(
        decision
    ).strip()


    if not decision:
        raise ValueError(
            "decision cannot be empty"
        )


    normalized_assumptions = (
        normalize_assumptions(
            assumptions
        )
    )


    if not normalized_assumptions:
        raise ValueError(
            "at least one valid assumption is required"
        )


    # ==================================
    # Build integrated evidence
    # ==================================

    evidence = (
        build_integrated_analysis_evidence(
            normalized_assumptions,

            financial_inputs=
                financial_inputs,

            top_company_analogues=
                top_company_analogues,

            top_failure_cases=
                top_failure_cases,
        )
    )


    # ==================================
    # Build summary
    # ==================================

    summary = (
        build_analysis_summary(
            evidence
        )
    )


    return {
        "decision":
            decision,

        "summary":
            summary,

        "assumptions":
            normalized_assumptions,

        "evidence":
            evidence,
    }

In [28]:
# CELL 26 - TEST FINAL ANALYZE_DECISION PIPELINE

test_decision = """
Launch an AI-powered SaaS platform for small businesses
that automates repetitive operational workflows.
""".strip()


final_analysis_result = (
    analyze_decision(
        decision=
            test_decision,

        assumptions=
            test_assumptions,

        financial_inputs=
            test_financial_inputs,

        top_company_analogues=5,

        top_failure_cases=5,
    )
)


print(
    "DECISION"
)

print(
    final_analysis_result[
        "decision"
    ]
)


print(
    "\nSUMMARY"
)

print(
    json.dumps(
        final_analysis_result[
            "summary"
        ],
        indent=2,
        ensure_ascii=False
    )
)


print(
    "\nASSUMPTIONS"
)

for assumption in final_analysis_result[
    "assumptions"
]:

    print(
        assumption
    )

DECISION
Launch an AI-powered SaaS platform for small businesses
that automates repetitive operational workflows.

SUMMARY
{
  "assumption_count": 2,
  "company_candidate_universe_size": 115798,
  "historical_failure_universe_size": 143,
  "contribution_margin": 0.3,
  "monthly_operating_profit": -30000000.0,
  "monthly_burn": 30000000.0,
  "runway_months": 16.666666666666668,
  "break_even_orders": 3000.0,
  "financial_flag_count": 1
}

ASSUMPTIONS
{'id': 'A1', 'assumption': 'Small businesses are willing to pay for AI-powered workflow automation software.', 'retrieval_query': 'Small AI SaaS companies providing workflow automation software for business customers.'}
{'id': 'A2', 'assumption': 'A small software team can build and operate a scalable SaaS platform.', 'retrieval_query': 'Small SaaS software companies building scalable business software with limited teams.'}


In [29]:
# CELL 27 - INSPECT FINAL BACKEND-READY ANALYSIS RESPONSE

final_analysis_json = (
    json.dumps(
        final_analysis_result,
        indent=2,
        ensure_ascii=False
    )
)


print(
    final_analysis_json[
        :20000
    ]
)

{
  "decision": "Launch an AI-powered SaaS platform for small businesses\nthat automates repetitive operational workflows.",
  "summary": {
    "assumption_count": 2,
    "company_candidate_universe_size": 115798,
    "historical_failure_universe_size": 143,
    "contribution_margin": 0.3,
    "monthly_operating_profit": -30000000.0,
    "monthly_burn": 30000000.0,
    "runway_months": 16.666666666666668,
    "break_even_orders": 3000.0,
    "financial_flag_count": 1
  },
  "assumptions": [
    {
      "id": "A1",
      "assumption": "Small businesses are willing to pay for AI-powered workflow automation software.",
      "retrieval_query": "Small AI SaaS companies providing workflow automation software for business customers."
    },
    {
      "id": "A2",
      "assumption": "A small software team can build and operate a scalable SaaS platform.",
      "retrieval_query": "Small SaaS software companies building scalable business software with limited teams."
    }
  ],
  "evidence": 

In [30]:
# CELL 28 - SETUP OPENROUTER ASSUMPTION ANALYZER

import os
import re
import requests


OPENROUTER_API_URL = (
    "https://openrouter.ai/api/v1/chat/completions"
)

OPENROUTER_MODEL = (
    "inclusionai/ling-3.0-flash-fin:free"
)


OPENROUTER_API_KEY = os.getenv(
    "OPENROUTER_API_KEY"
)


print(
    "OpenRouter API key:",
    "FOUND"
    if OPENROUTER_API_KEY
    else "MISSING"
)

print(
    "Model:",
    OPENROUTER_MODEL
)

OpenRouter API key: FOUND
Model: inclusionai/ling-3.0-flash-fin:free


In [31]:
# CELL 29 - DEFINE SAFE LLM JSON PARSER

def parse_llm_json(
    text
):
    """
    Parse JSON returned by the LLM.

    Handles:
    - pure JSON
    - ```json ... ```
    - extra text around JSON object
    """

    if not isinstance(
        text,
        str
    ):
        raise ValueError(
            "LLM response must be a string."
        )


    cleaned = text.strip()


    # Remove markdown code fences
    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE
    )

    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned
    )


    # First attempt: direct JSON
    try:

        return json.loads(
            cleaned
        )

    except json.JSONDecodeError:
        pass


    # Second attempt:
    # extract first JSON object
    start = cleaned.find(
        "{"
    )

    end = cleaned.rfind(
        "}"
    )


    if (
        start == -1
        or end == -1
        or end <= start
    ):
        raise ValueError(
            "No valid JSON object found "
            "in LLM response."
        )


    json_text = cleaned[
        start:end + 1
    ]


    return json.loads(
        json_text
    )

In [34]:
# CELL 30 - DEFINE LLM ASSUMPTION ANALYZER

def analyze_decision_assumptions(
    decision,
    *,
    min_assumptions=3,
    max_assumptions=5,
):
    """
    Extract decision assumptions and semantic
    retrieval queries using OpenRouter.

    The LLM is used for interpretation only.
    Evidence and financial calculations are handled
    separately by deterministic/retrieval components.
    """

    if not OPENROUTER_API_KEY:

        raise ValueError(
            "OPENROUTER_API_KEY is not available."
        )


    decision = str(
        decision
    ).strip()


    if not decision:

        raise ValueError(
            "decision cannot be empty"
        )


    system_prompt = f"""
You are an assumption extraction component for an
AI decision stress-testing platform.

Your task is to identify the most important assumptions
embedded in a user's business or strategic decision.

Extract between {min_assumptions} and {max_assumptions}
assumptions.

Rules:

1. Do not invent facts that are not implied by the user.
2. Do not assign probabilities.
3. Do not invent financial figures.
4. Each assumption must be falsifiable or testable.
5. Focus on assumptions that could materially affect
   whether the decision succeeds or fails.
6. Keep assumptions distinct from one another.
7. For every assumption, create a retrieval_query.
8. retrieval_query should describe the type of company,
   business model, market behavior, or failure mechanism
   that would provide useful historical analogues.
9. retrieval_query should be search-friendly and
   semantically descriptive.
10. Return JSON only.

Required JSON schema:

{{
  "assumptions": [
    {{
      "id": "A1",
      "assumption": "string",
      "retrieval_query": "string"
    }}
  ]
}}
""".strip()


    user_prompt = f"""
Decision to stress-test:

{decision}

Identify the key assumptions underlying this decision.
""".strip()


    response = requests.post(
        OPENROUTER_API_URL,

        headers={
            "Authorization":
                f"Bearer {OPENROUTER_API_KEY}",

            "Content-Type":
                "application/json",
        },

        json={
            "model":
                OPENROUTER_MODEL,

            "messages": [
                {
                    "role":
                        "system",

                    "content":
                        system_prompt,
                },
                {
                    "role":
                        "user",

                    "content":
                        user_prompt,
                },
            ],

            "temperature":
                0.2,
        },

        timeout=120,
    )


    response.raise_for_status()


    response_json = (
        response.json()
    )


    raw_content = (
        response_json[
            "choices"
        ][0][
            "message"
        ][
            "content"
        ]
    )


    parsed = parse_llm_json(
        raw_content
    )


    normalized = (
        normalize_assumptions(
            parsed.get(
                "assumptions",
                []
            )
        )
    )


    if not normalized:

        raise ValueError(
            "LLM returned no valid assumptions."
        )


    return {
        "decision":
            decision,

        "assumptions":
            normalized,

        "raw_response":
            raw_content,
    }

In [35]:
# CELL 31 - DEFINE FULL USER DECISION PIPELINE

def analyze_user_decision(
    *,
    decision,
    financial_inputs=None,
    top_company_analogues=5,
    top_failure_cases=5,
):
    """
    Complete end-to-end decision analysis pipeline.

    Flow:
    decision
        -> LLM assumption extraction
        -> qualitative evidence retrieval
        -> deterministic financial stress test
        -> IDX directional benchmark
        -> backend-ready response
    """

    pipeline_start = (
        time.perf_counter()
    )


    # ==================================
    # 1. Extract assumptions with LLM
    # ==================================

    assumption_analysis = (
        analyze_decision_assumptions(
            decision
        )
    )


    assumptions = (
        assumption_analysis[
            "assumptions"
        ]
    )


    # ==================================
    # 2. Run evidence pipeline
    # ==================================

    analysis = (
        analyze_decision(
            decision=
                decision,

            assumptions=
                assumptions,

            financial_inputs=
                financial_inputs,

            top_company_analogues=
                top_company_analogues,

            top_failure_cases=
                top_failure_cases,
        )
    )


    elapsed_seconds = (
        time.perf_counter()
        - pipeline_start
    )


    # ==================================
    # 3. Final backend-ready response
    # ==================================

    return {
        "decision":
            decision,

        "processing_time_seconds":
            round(
                elapsed_seconds,
                3
            ),

        "assumption_analysis": {
            "assumption_count":
                len(
                    assumptions
                ),

            "assumptions":
                assumptions,
        },

        "summary":
            analysis[
                "summary"
            ],

        "evidence":
            analysis[
                "evidence"
            ],
    }

In [36]:
# CELL 32 - TEST FULL END-TO-END USER DECISION PIPELINE

real_test_decision = """
Launch an AI-powered SaaS platform for small businesses
that automates repetitive operational workflows.
""".strip()


real_test_financial_inputs = {
    "monthly_orders":
        2_000,

    "revenue_per_order":
        100_000,

    "variable_cost_per_order":
        45_000,

    "promo_subsidy_per_order":
        10_000,

    "delivery_cost_per_order":
        15_000,

    "monthly_fixed_cost":
        80_000_000,

    "monthly_driver_cost":
        10_000_000,

    "cash_balance":
        500_000_000,
}


full_user_analysis = (
    analyze_user_decision(
        decision=
            real_test_decision,

        financial_inputs=
            real_test_financial_inputs,

        top_company_analogues=5,

        top_failure_cases=5,
    )
)


print(
    "FULL END-TO-END ANALYSIS COMPLETE"
)

print(
    "Processing time:",
    full_user_analysis[
        "processing_time_seconds"
    ],
    "seconds"
)


print(
    "\nASSUMPTIONS GENERATED:"
)


for assumption in full_user_analysis[
    "assumption_analysis"
][
    "assumptions"
]:

    print(
        "\n",
        assumption[
            "id"
        ],
        "-",
        assumption[
            "assumption"
        ]
    )

    print(
        "Retrieval query:",
        assumption[
            "retrieval_query"
        ]
    )

FULL END-TO-END ANALYSIS COMPLETE
Processing time: 3.418 seconds

ASSUMPTIONS GENERATED:

 A1 - Small businesses have sufficient willingness and budget to pay a recurring SaaS subscription for AI-powered workflow automation, rather than relying on free or manual alternatives.
Retrieval query: small business adoption of paid SaaS automation tools willingness to pay subscription pricing

 A2 - The AI technology can reliably and accurately automate a meaningful share of repetitive operational workflows across diverse small business types without requiring extensive customization per client.
Retrieval query: AI workflow automation accuracy generalization across small business verticals low-code customization

 A3 - Small businesses possess adequate digital maturity, structured operational data, and integration readiness to onboard and effectively use a cloud-based SaaS platform.
Retrieval query: small business digital maturity cloud SaaS adoption data readiness integration capability

 A4 

In [37]:
# CELL 33 - INSPECT END-TO-END SUMMARY

print(
    json.dumps(
        full_user_analysis[
            "summary"
        ],
        indent=2,
        ensure_ascii=False
    )
)

{
  "assumption_count": 5,
  "company_candidate_universe_size": 115798,
  "historical_failure_universe_size": 143,
  "contribution_margin": 0.3,
  "monthly_operating_profit": -30000000.0,
  "monthly_burn": 30000000.0,
  "runway_months": 16.666666666666668,
  "break_even_orders": 3000.0,
  "financial_flag_count": 1
}


In [38]:
# CELL 34 - INSPECT END-TO-END BACKEND RESPONSE

full_user_analysis_json = (
    json.dumps(
        full_user_analysis,
        indent=2,
        ensure_ascii=False
    )
)


print(
    full_user_analysis_json[
        :30000
    ]
)

{
  "decision": "Launch an AI-powered SaaS platform for small businesses\nthat automates repetitive operational workflows.",
  "processing_time_seconds": 3.418,
  "assumption_analysis": {
    "assumption_count": 5,
    "assumptions": [
      {
        "id": "A1",
        "assumption": "Small businesses have sufficient willingness and budget to pay a recurring SaaS subscription for AI-powered workflow automation, rather than relying on free or manual alternatives.",
        "retrieval_query": "small business adoption of paid SaaS automation tools willingness to pay subscription pricing"
      },
      {
        "id": "A2",
        "assumption": "The AI technology can reliably and accurately automate a meaningful share of repetitive operational workflows across diverse small business types without requiring extensive customization per client.",
        "retrieval_query": "AI workflow automation accuracy generalization across small business verticals low-code customization"
      },
     

In [39]:
# CELL 35 - BUILD STRICT EVIDENCE PACKET FOR LLM SYNTHESIS

def build_assumption_evidence_packet(
    assumption_item
):
    """
    Convert one integrated qualitative-evidence item
    into a compact, grounded packet for the LLM.

    Only fields already present in retrieved evidence
    are included.
    """

    company_evidence = []

    for company in assumption_item.get(
        "company_analogues",
        []
    ):
        company_evidence.append(
            {
                "rank":
                    company.get("rank"),

                "name":
                    company.get("name"),

                "semantic_score":
                    company.get("semantic_score"),

                "short_description":
                    company.get(
                        "short_description"
                    ),

                "categories":
                    company.get("categories"),

                "locations":
                    company.get("locations"),

                "employee_range":
                    company.get(
                        "employee_range"
                    ),

                "founded_year":
                    company.get(
                        "founded_year"
                    ),
            }
        )


    failure_evidence = []

    for failure in assumption_item.get(
        "historical_failures",
        []
    ):
        failure_evidence.append(
            {
                "rank":
                    failure.get("rank"),

                "company":
                    failure.get("company"),

                "semantic_score":
                    failure.get(
                        "semantic_score"
                    ),

                "funding":
                    failure.get("funding"),

                "category":
                    failure.get("category"),

                "investors":
                    failure.get("investors"),

                "failure_reason":
                    failure.get(
                        "failure_reason"
                    ),

                "prompt":
                    failure.get("prompt"),
            }
        )


    return {
        "assumption_id":
            assumption_item.get(
                "assumption_id"
            ),

        "assumption":
            assumption_item.get(
                "assumption"
            ),

        "retrieval_query":
            assumption_item.get(
                "retrieval_query"
            ),

        "company_analogues":
            company_evidence,

        "historical_failures":
            failure_evidence,
    }

In [40]:
# CELL 36 - DEFINE STRICT GROUNDED ASSUMPTION EVALUATOR

def evaluate_assumption_with_evidence(
    assumption_packet
):
    """
    Use Ling only to interpret retrieved evidence.

    The model is explicitly forbidden from:
    - inventing facts
    - inventing probabilities
    - inventing financial numbers
    - citing companies/evidence not included
    """

    if not OPENROUTER_API_KEY:
        raise ValueError(
            "OPENROUTER_API_KEY is not available."
        )


    system_prompt = """
You are an evidence-grounded decision stress-test evaluator.

You MUST base every conclusion only on the evidence
provided in the user message.

STRICT RULES:

1. Do not use outside knowledge.
2. Do not invent companies, events, market facts,
   statistics, financial values, probabilities,
   benchmarks, thresholds, or causal claims.
3. Do not infer facts that are not directly supported
   by the provided evidence.
4. Do not treat semantic similarity score as a
   probability or success/failure likelihood.
5. Do not claim that a company succeeded or failed
   unless the provided evidence explicitly says so.
6. Historical failure evidence may be used only for
   failure mechanisms explicitly stated in
   failure_reason.
7. Company analogue evidence may be used only to show
   similarity in business type, category, product,
   location, company size, founding year, or other
   fields explicitly provided.
8. If the evidence is weak, mixed, or insufficient,
   explicitly say so.
9. Do not create exact risk probabilities.
10. Do not recommend conclusions beyond what the
    evidence supports.
11. Every important observation must include evidence
    references using only the IDs supplied below.
12. Return JSON only.

Allowed assessment values:

- "supported"
- "partially_supported"
- "challenged"
- "mixed"
- "insufficient_evidence"

Required JSON schema:

{
  "assessment": "one allowed value",
  "assessment_summary": "short grounded explanation",
  "supporting_signals": [
    {
      "statement": "grounded statement",
      "evidence_refs": ["COMPANY_1"]
    }
  ],
  "risk_signals": [
    {
      "statement": "grounded statement",
      "evidence_refs": ["FAILURE_1"]
    }
  ],
  "failure_mechanisms": [
    {
      "mechanism": "failure mechanism explicitly supported by failure evidence",
      "evidence_refs": ["FAILURE_1"]
    }
  ],
  "evidence_gaps": [
    "what cannot be concluded from available evidence"
  ]
}
""".strip()


    # ----------------------------------
    # Add explicit IDs for citations
    # ----------------------------------

    company_with_refs = []

    for index, company in enumerate(
        assumption_packet[
            "company_analogues"
        ],
        start=1
    ):

        company_with_refs.append(
            {
                "evidence_ref":
                    f"COMPANY_{index}",

                **company
            }
        )


    failures_with_refs = []

    for index, failure in enumerate(
        assumption_packet[
            "historical_failures"
        ],
        start=1
    ):

        failures_with_refs.append(
            {
                "evidence_ref":
                    f"FAILURE_{index}",

                **failure
            }
        )


    evidence_payload = {
        "assumption_id":
            assumption_packet[
                "assumption_id"
            ],

        "assumption":
            assumption_packet[
                "assumption"
            ],

        "retrieval_query":
            assumption_packet[
                "retrieval_query"
            ],

        "company_analogues":
            company_with_refs,

        "historical_failures":
            failures_with_refs,
    }


    user_prompt = (
        "Evaluate the assumption using ONLY the "
        "following evidence.\n\n"
        + json.dumps(
            evidence_payload,
            indent=2,
            ensure_ascii=False
        )
    )


    response = requests.post(
        OPENROUTER_API_URL,

        headers={
            "Authorization":
                f"Bearer {OPENROUTER_API_KEY}",

            "Content-Type":
                "application/json",
        },

        json={
            "model":
                OPENROUTER_MODEL,

            "messages": [
                {
                    "role":
                        "system",

                    "content":
                        system_prompt,
                },
                {
                    "role":
                        "user",

                    "content":
                        user_prompt,
                },
            ],

            "temperature":
                0.0,
        },

        timeout=120,
    )


    response.raise_for_status()


    raw_content = (
        response.json()[
            "choices"
        ][0][
            "message"
        ][
            "content"
        ]
    )


    parsed = parse_llm_json(
        raw_content
    )


    return {
        "assumption_id":
            assumption_packet[
                "assumption_id"
            ],

        "assumption":
            assumption_packet[
                "assumption"
            ],

        "evaluation":
            parsed,

        "raw_response":
            raw_content,
    }

In [41]:
# CELL 37 - RUN GROUNDED SYNTHESIS FOR ALL ASSUMPTIONS

qualitative_assumptions = (
    full_user_analysis[
        "evidence"
    ][
        "qualitative_evidence"
    ][
        "assumptions"
    ]
)


grounded_assumption_evaluations = []


for assumption_item in qualitative_assumptions:

    packet = (
        build_assumption_evidence_packet(
            assumption_item
        )
    )


    evaluation = (
        evaluate_assumption_with_evidence(
            packet
        )
    )


    grounded_assumption_evaluations.append(
        evaluation
    )


print(
    "Assumptions evaluated:",
    len(
        grounded_assumption_evaluations
    )
)


for item in grounded_assumption_evaluations:

    print(
        "\n" + "=" * 100
    )

    print(
        item[
            "assumption_id"
        ],
        "-",
        item[
            "assumption"
        ]
    )


    print(
        json.dumps(
            item[
                "evaluation"
            ],
            indent=2,
            ensure_ascii=False
        )
    )

Assumptions evaluated: 5

A1 - Small businesses have sufficient willingness and budget to pay a recurring SaaS subscription for AI-powered workflow automation, rather than relying on free or manual alternatives.
{
  "assessment": "insufficient_evidence",
  "assessment_summary": "The provided evidence does not contain direct data on small businesses' willingness or budget to pay for AI-powered workflow automation SaaS subscriptions. The company analogues are SaaS companies themselves but do not specifically address AI-powered workflow automation or small business customer behavior. The historical failures are unrelated to this specific assumption about small business SaaS adoption.",
  "supporting_signals": [
    {
      "statement": "Multiple SaaS companies exist across various domains (payments, retail, cost control), indicating a general SaaS market presence.",
      "evidence_refs": [
        "COMPANY_1",
        "COMPANY_2",
        "COMPANY_3",
        "COMPANY_4",
        "COMPAN

In [42]:
# CELL 38 - ATTACH GROUNDED SYNTHESIS TO FINAL ANALYSIS

final_grounded_analysis = (
    full_user_analysis.copy()
)


final_grounded_analysis[
    "assumption_evaluations"
] = grounded_assumption_evaluations


print(
    "FINAL GROUNDED ANALYSIS READY"
)

print(
    "Decision:",
    final_grounded_analysis[
        "decision"
    ]
)

print(
    "Assumption evaluations:",
    len(
        final_grounded_analysis[
            "assumption_evaluations"
        ]
    )
)

FINAL GROUNDED ANALYSIS READY
Decision: Launch an AI-powered SaaS platform for small businesses
that automates repetitive operational workflows.
Assumption evaluations: 5


In [43]:
# CELL 39 - VALIDATE GROUNDED EVALUATION STRUCTURE

ALLOWED_ASSESSMENTS = {
    "supported",
    "partially_supported",
    "challenged",
    "mixed",
    "insufficient_evidence",
}


def validate_grounded_evaluation(
    assumption_packet,
    evaluation,
):
    """
    Deterministically validate evaluation structure
    and evidence references.

    This validator does NOT judge semantic truth yet.
    It ensures Ling can only cite evidence that actually
    exists in the provided packet.
    """

    errors = []
    warnings = []


    # ==================================
    # Valid evidence references
    # ==================================

    valid_company_refs = {
        f"COMPANY_{index}"
        for index in range(
            1,
            len(
                assumption_packet.get(
                    "company_analogues",
                    []
                )
            ) + 1
        )
    }


    valid_failure_refs = {
        f"FAILURE_{index}"
        for index in range(
            1,
            len(
                assumption_packet.get(
                    "historical_failures",
                    []
                )
            ) + 1
        )
    }


    valid_refs = (
        valid_company_refs
        |
        valid_failure_refs
    )


    # ==================================
    # Assessment value
    # ==================================

    assessment = evaluation.get(
        "assessment"
    )


    if assessment not in ALLOWED_ASSESSMENTS:

        errors.append(
            f"Invalid assessment: {assessment}"
        )


    # ==================================
    # Required fields
    # ==================================

    required_list_fields = [
        "supporting_signals",
        "risk_signals",
        "failure_mechanisms",
        "evidence_gaps",
    ]


    for field in required_list_fields:

        if field not in evaluation:

            errors.append(
                f"Missing field: {field}"
            )

        elif not isinstance(
            evaluation[field],
            list
        ):

            errors.append(
                f"{field} must be a list"
            )


    # ==================================
    # Check evidence references
    # ==================================

    reference_fields = [
        "supporting_signals",
        "risk_signals",
        "failure_mechanisms",
    ]


    for field in reference_fields:

        for item_index, item in enumerate(
            evaluation.get(
                field,
                []
            ),
            start=1
        ):

            refs = item.get(
                "evidence_refs",
                []
            )


            if not refs:

                warnings.append(
                    f"{field}[{item_index}] "
                    "has no evidence_refs"
                )

                continue


            for ref in refs:

                if ref not in valid_refs:

                    errors.append(
                        f"{field}[{item_index}] "
                        f"uses invalid evidence ref: {ref}"
                    )


                # Failure mechanisms should only
                # cite historical failure evidence
                if (
                    field
                    == "failure_mechanisms"
                    and ref not in valid_failure_refs
                ):

                    errors.append(
                        f"{field}[{item_index}] "
                        f"must cite FAILURE_* only: {ref}"
                    )


    return {
        "is_valid":
            len(errors) == 0,

        "errors":
            errors,

        "warnings":
            warnings,

        "valid_company_refs":
            sorted(
                valid_company_refs
            ),

        "valid_failure_refs":
            sorted(
                valid_failure_refs
            ),
    }

In [44]:
# CELL 40 - VALIDATE ALL CURRENT GROUNDED EVALUATIONS

grounding_validation_results = []


for assumption_item, evaluation_item in zip(
    qualitative_assumptions,
    grounded_assumption_evaluations,
):

    packet = (
        build_assumption_evidence_packet(
            assumption_item
        )
    )


    validation = (
        validate_grounded_evaluation(
            packet,
            evaluation_item[
                "evaluation"
            ]
        )
    )


    grounding_validation_results.append(
        {
            "assumption_id":
                evaluation_item[
                    "assumption_id"
                ],

            **validation,
        }
    )


for result in grounding_validation_results:

    print(
        "\n" + "=" * 80
    )

    print(
        "Assumption:",
        result[
            "assumption_id"
        ]
    )

    print(
        "Valid:",
        result[
            "is_valid"
        ]
    )

    print(
        "Errors:",
        result[
            "errors"
        ]
    )

    print(
        "Warnings:",
        result[
            "warnings"
        ]
    )


Assumption: A1
Valid: True
Errors: []
Warnings: []

Assumption: A2
Valid: True
Errors: []
Warnings: []

Assumption: A3
Valid: True
Errors: []
Warnings: []

Assumption: A4
Valid: True
Errors: []
Warnings: []

Assumption: A5
Valid: True
Errors: []
Warnings: []


In [45]:
# CELL 41 - DEFINE STRICT CLAIM GROUNDING REVIEWER

def review_evaluation_grounding(
    assumption_packet,
    evaluation,
):
    """
    Second-pass grounding review.

    The reviewer may ONLY:
    - keep a supported claim,
    - make it more conservative,
    - remove unsupported claims.

    It may NOT add new evidence or analysis.
    """

    company_with_refs = []

    for index, company in enumerate(
        assumption_packet[
            "company_analogues"
        ],
        start=1
    ):

        company_with_refs.append(
            {
                "evidence_ref":
                    f"COMPANY_{index}",

                **company,
            }
        )


    failures_with_refs = []

    for index, failure in enumerate(
        assumption_packet[
            "historical_failures"
        ],
        start=1
    ):

        failures_with_refs.append(
            {
                "evidence_ref":
                    f"FAILURE_{index}",

                **failure,
            }
        )


    review_payload = {
        "assumption":
            assumption_packet[
                "assumption"
            ],

        "evidence": {
            "company_analogues":
                company_with_refs,

            "historical_failures":
                failures_with_refs,
        },

        "evaluation_to_review":
            evaluation,
    }


    system_prompt = """
You are a strict evidence-grounding reviewer.

Your ONLY task is to remove or weaken claims that are
not directly supported by the supplied evidence.

Do NOT add any new analysis.

STRICT RULES:

1. Use only the supplied evidence.
2. Do not use outside knowledge.
3. Do not add new facts, companies, mechanisms,
   statistics, probabilities, or explanations.
4. A claim is valid only if the cited evidence directly
   supports the factual content of that claim.
5. Company size, founding year, category, or description
   must NOT be converted into claims about traction,
   success, market share, dominance, adoption, financial
   strength, or product-market fit unless explicitly
   stated in the evidence.
6. Funding amount must NOT be converted into a claim
   about success, demand, traction, or product quality.
7. Historical failure mechanisms must reflect only
   what is explicitly stated in failure_reason.
8. If a claim contains both supported and unsupported
   content, rewrite it to retain only the supported part.
9. If nothing remains supported, remove the claim.
10. Preserve evidence_refs only when they directly
    support the retained claim.
11. Do not make the assessment stronger than the
    evidence allows.
12. When direct evidence is absent, prefer
    "insufficient_evidence".
13. Return JSON only, using exactly the same schema
    as the evaluation being reviewed.
""".strip()


    response = requests.post(
        OPENROUTER_API_URL,

        headers={
            "Authorization":
                f"Bearer {OPENROUTER_API_KEY}",

            "Content-Type":
                "application/json",
        },

        json={
            "model":
                OPENROUTER_MODEL,

            "messages": [
                {
                    "role":
                        "system",

                    "content":
                        system_prompt,
                },
                {
                    "role":
                        "user",

                    "content":
                        json.dumps(
                            review_payload,
                            indent=2,
                            ensure_ascii=False
                        ),
                },
            ],

            "temperature":
                0.0,
        },

        timeout=120,
    )


    response.raise_for_status()


    raw_content = (
        response.json()[
            "choices"
        ][0][
            "message"
        ][
            "content"
        ]
    )


    reviewed = parse_llm_json(
        raw_content
    )


    return reviewed

In [46]:
# CELL 42 - RUN STRICT GROUNDING REVIEW FOR ALL ASSUMPTIONS

reviewed_assumption_evaluations = []


for assumption_item, evaluation_item in zip(
    qualitative_assumptions,
    grounded_assumption_evaluations,
):

    packet = (
        build_assumption_evidence_packet(
            assumption_item
        )
    )


    reviewed_evaluation = (
        review_evaluation_grounding(
            packet,
            evaluation_item[
                "evaluation"
            ]
        )
    )


    reviewed_assumption_evaluations.append(
        {
            "assumption_id":
                evaluation_item[
                    "assumption_id"
                ],

            "assumption":
                evaluation_item[
                    "assumption"
                ],

            "evaluation":
                reviewed_evaluation,
        }
    )


print(
    "Reviewed assumptions:",
    len(
        reviewed_assumption_evaluations
    )
)


for item in reviewed_assumption_evaluations:

    print(
        "\n" + "=" * 100
    )

    print(
        item[
            "assumption_id"
        ],
        "-",
        item[
            "assumption"
        ]
    )

    print(
        json.dumps(
            item[
                "evaluation"
            ],
            indent=2,
            ensure_ascii=False
        )
    )

Reviewed assumptions: 5

A1 - Small businesses have sufficient willingness and budget to pay a recurring SaaS subscription for AI-powered workflow automation, rather than relying on free or manual alternatives.
{
  "assessment": "insufficient_evidence",
  "assessment_summary": "The provided evidence does not contain direct data on small businesses' willingness or budget to pay for AI-powered workflow automation SaaS subscriptions. The company analogues are SaaS companies themselves but do not specifically address AI-powered workflow automation or small business customer behavior. The historical failures are unrelated to this specific assumption about small business SaaS adoption.",
  "supporting_signals": [
    {
      "statement": "Multiple SaaS companies exist across various domains (payments, retail, cost control), indicating a general SaaS market presence.",
      "evidence_refs": [
        "COMPANY_1",
        "COMPANY_2",
        "COMPANY_3",
        "COMPANY_4",
        "COMPANY

In [47]:
# CELL 43 - VALIDATE REVIEWED EVALUATIONS

reviewed_validation_results = []


for assumption_item, reviewed_item in zip(
    qualitative_assumptions,
    reviewed_assumption_evaluations,
):

    packet = (
        build_assumption_evidence_packet(
            assumption_item
        )
    )


    validation = (
        validate_grounded_evaluation(
            packet,
            reviewed_item[
                "evaluation"
            ]
        )
    )


    reviewed_validation_results.append(
        {
            "assumption_id":
                reviewed_item[
                    "assumption_id"
                ],

            **validation,
        }
    )


all_reviewed_valid = all(
    item[
        "is_valid"
    ]
    for item in reviewed_validation_results
)


for result in reviewed_validation_results:

    print(
        "\n" + "=" * 80
    )

    print(
        "Assumption:",
        result[
            "assumption_id"
        ]
    )

    print(
        "Valid:",
        result[
            "is_valid"
        ]
    )

    print(
        "Errors:",
        result[
            "errors"
        ]
    )

    print(
        "Warnings:",
        result[
            "warnings"
        ]
    )


print(
    "\nALL REVIEWED EVALUATIONS VALID:",
    all_reviewed_valid
)


Assumption: A1
Valid: True
Errors: []
Warnings: []

Assumption: A2
Valid: True
Errors: []
Warnings: []

Assumption: A3
Valid: True
Errors: []
Warnings: []

Assumption: A4
Valid: True
Errors: []
Warnings: []

Assumption: A5
Valid: True
Errors: []
Warnings: []

ALL REVIEWED EVALUATIONS VALID: True


In [48]:
# CELL 44 - COMPARE ORIGINAL VS REVIEWED EVALUATIONS

for original, reviewed in zip(
    grounded_assumption_evaluations,
    reviewed_assumption_evaluations,
):

    print(
        "\n" + "#" * 100
    )

    print(
        "ASSUMPTION:",
        original[
            "assumption_id"
        ]
    )


    print(
        "\nBEFORE REVIEW"
    )

    print(
        json.dumps(
            original[
                "evaluation"
            ],
            indent=2,
            ensure_ascii=False
        )
    )


    print(
        "\nAFTER REVIEW"
    )

    print(
        json.dumps(
            reviewed[
                "evaluation"
            ],
            indent=2,
            ensure_ascii=False
        )
    )


####################################################################################################
ASSUMPTION: A1

BEFORE REVIEW
{
  "assessment": "insufficient_evidence",
  "assessment_summary": "The provided evidence does not contain direct data on small businesses' willingness or budget to pay for AI-powered workflow automation SaaS subscriptions. The company analogues are SaaS companies themselves but do not specifically address AI-powered workflow automation or small business customer behavior. The historical failures are unrelated to this specific assumption about small business SaaS adoption.",
  "supporting_signals": [
    {
      "statement": "Multiple SaaS companies exist across various domains (payments, retail, cost control), indicating a general SaaS market presence.",
      "evidence_refs": [
        "COMPANY_1",
        "COMPANY_2",
        "COMPANY_3",
        "COMPANY_4",
        "COMPANY_5"
      ]
    },
    {
      "statement": "Some analogues serve small busines

In [49]:
# CELL 45 - SETUP LLM CACHE

import hashlib


LLM_CACHE_FILE = Path(
    "data/outputs/llm_cache.json"
)


def load_llm_cache():
    """
    Load persistent LLM cache from disk.
    """

    if not LLM_CACHE_FILE.exists():
        return {}


    with open(
        LLM_CACHE_FILE,
        "r",
        encoding="utf-8"
    ) as file:

        return json.load(
            file
        )


def save_llm_cache(
    cache
):
    """
    Persist LLM cache to disk.
    """

    LLM_CACHE_FILE.parent.mkdir(
        parents=True,
        exist_ok=True
    )


    with open(
        LLM_CACHE_FILE,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            cache,
            file,
            indent=2,
            ensure_ascii=False
        )


llm_cache = load_llm_cache()


print(
    "LLM cache entries:",
    len(
        llm_cache
    )
)

print(
    "Cache file:",
    LLM_CACHE_FILE
)

LLM cache entries: 0
Cache file: data\llm_cache.json


In [50]:
# CELL 46 - DEFINE CACHED OPENROUTER REQUEST

def build_llm_cache_key(
    *,
    model,
    system_prompt,
    user_prompt,
    temperature,
):
    payload = {
        "model":
            model,

        "system_prompt":
            system_prompt,

        "user_prompt":
            user_prompt,

        "temperature":
            temperature,
    }


    canonical = json.dumps(
        payload,
        sort_keys=True,
        ensure_ascii=False
    )


    return hashlib.sha256(
        canonical.encode(
            "utf-8"
        )
    ).hexdigest()


def call_openrouter_cached(
    *,
    system_prompt,
    user_prompt,
    temperature=0.0,
):
    """
    Call OpenRouter with persistent file cache.

    Identical prompts use cached responses and
    do not consume another API request.
    """

    cache_key = build_llm_cache_key(
        model=
            OPENROUTER_MODEL,

        system_prompt=
            system_prompt,

        user_prompt=
            user_prompt,

        temperature=
            temperature,
    )


    # ==================================
    # Cache hit
    # ==================================

    if cache_key in llm_cache:

        print(
            "LLM CACHE HIT"
        )

        return llm_cache[
            cache_key
        ]


    # ==================================
    # Cache miss → real API request
    # ==================================

    print(
        "LLM CACHE MISS → calling OpenRouter"
    )


    response = requests.post(
        OPENROUTER_API_URL,

        headers={
            "Authorization":
                f"Bearer {OPENROUTER_API_KEY}",

            "Content-Type":
                "application/json",
        },

        json={
            "model":
                OPENROUTER_MODEL,

            "messages": [
                {
                    "role":
                        "system",

                    "content":
                        system_prompt,
                },
                {
                    "role":
                        "user",

                    "content":
                        user_prompt,
                },
            ],

            "temperature":
                temperature,
        },

        timeout=120,
    )


    response.raise_for_status()


    raw_content = (
        response.json()[
            "choices"
        ][0][
            "message"
        ][
            "content"
        ]
    )


    # ==================================
    # Save result
    # ==================================

    llm_cache[
        cache_key
    ] = raw_content


    save_llm_cache(
        llm_cache
    )


    return raw_content

In [51]:
# CELL 47 - TEST LLM CACHE

test_cache_system_prompt = """
Return JSON only.
""".strip()


test_cache_user_prompt = """
Return exactly:

{
  "status": "ok"
}
""".strip()


test_cache_response_1 = (
    call_openrouter_cached(
        system_prompt=
            test_cache_system_prompt,

        user_prompt=
            test_cache_user_prompt,

        temperature=0.0,
    )
)


print(
    "\nFIRST RESULT:"
)

print(
    test_cache_response_1
)


test_cache_response_2 = (
    call_openrouter_cached(
        system_prompt=
            test_cache_system_prompt,

        user_prompt=
            test_cache_user_prompt,

        temperature=0.0,
    )
)


print(
    "\nSECOND RESULT:"
)

print(
    test_cache_response_2
)

LLM CACHE MISS → calling OpenRouter

FIRST RESULT:
{
  "status": "ok"
}
LLM CACHE HIT

SECOND RESULT:
{
  "status": "ok"
}


In [52]:
# CELL 48 - CHECK CURRENT LLM CACHE

print(
    "Current cache entries:",
    len(llm_cache)
)

print(
    "Cache file exists:",
    LLM_CACHE_FILE.exists()
)

Current cache entries: 1
Cache file exists: True


In [53]:
# CELL 49 - REDEFINE ASSUMPTION ANALYZER WITH CACHE

def analyze_decision_assumptions(
    decision,
    *,
    min_assumptions=3,
    max_assumptions=5,
):
    """
    Cached assumption extraction using OpenRouter.
    """

    if not OPENROUTER_API_KEY:
        raise ValueError(
            "OPENROUTER_API_KEY is not available."
        )


    decision = str(
        decision
    ).strip()


    if not decision:
        raise ValueError(
            "decision cannot be empty"
        )


    system_prompt = f"""
You are an assumption extraction component for an
AI decision stress-testing platform.

Your task is to identify the most important assumptions
embedded in a user's business or strategic decision.

Extract between {min_assumptions} and {max_assumptions}
assumptions.

Rules:

1. Do not invent facts that are not implied by the user.
2. Do not assign probabilities.
3. Do not invent financial figures.
4. Each assumption must be falsifiable or testable.
5. Focus on assumptions that could materially affect
   whether the decision succeeds or fails.
6. Keep assumptions distinct from one another.
7. For every assumption, create a retrieval_query.
8. retrieval_query should describe the type of company,
   business model, market behavior, or failure mechanism
   that would provide useful historical analogues.
9. retrieval_query should be search-friendly and
   semantically descriptive.
10. Return JSON only.

Required JSON schema:

{{
  "assumptions": [
    {{
      "id": "A1",
      "assumption": "string",
      "retrieval_query": "string"
    }}
  ]
}}
""".strip()


    user_prompt = f"""
Decision to stress-test:

{decision}

Identify the key assumptions underlying this decision.
""".strip()


    raw_content = (
        call_openrouter_cached(
            system_prompt=
                system_prompt,

            user_prompt=
                user_prompt,

            temperature=0.2,
        )
    )


    parsed = (
        parse_llm_json(
            raw_content
        )
    )


    normalized = (
        normalize_assumptions(
            parsed.get(
                "assumptions",
                []
            )
        )
    )


    if not normalized:
        raise ValueError(
            "LLM returned no valid assumptions."
        )


    return {
        "decision":
            decision,

        "assumptions":
            normalized,

        "raw_response":
            raw_content,
    }

In [54]:
# CELL 50 - REDEFINE GROUNDED EVALUATOR WITH CACHE

def evaluate_assumption_with_evidence(
    assumption_packet
):
    """
    Cached evidence-grounded evaluator.
    """

    company_with_refs = []

    for index, company in enumerate(
        assumption_packet[
            "company_analogues"
        ],
        start=1
    ):

        company_with_refs.append(
            {
                "evidence_ref":
                    f"COMPANY_{index}",

                **company
            }
        )


    failures_with_refs = []

    for index, failure in enumerate(
        assumption_packet[
            "historical_failures"
        ],
        start=1
    ):

        failures_with_refs.append(
            {
                "evidence_ref":
                    f"FAILURE_{index}",

                **failure
            }
        )


    evidence_payload = {
        "assumption_id":
            assumption_packet[
                "assumption_id"
            ],

        "assumption":
            assumption_packet[
                "assumption"
            ],

        "retrieval_query":
            assumption_packet[
                "retrieval_query"
            ],

        "company_analogues":
            company_with_refs,

        "historical_failures":
            failures_with_refs,
    }


    system_prompt = """
You are an evidence-grounded decision stress-test evaluator.

You MUST base every conclusion only on the evidence
provided in the user message.

STRICT RULES:

1. Do not use outside knowledge.
2. Do not invent companies, events, market facts,
   statistics, financial values, probabilities,
   benchmarks, thresholds, or causal claims.
3. Do not infer facts that are not directly supported
   by the provided evidence.
4. Do not treat semantic similarity score as a
   probability or success/failure likelihood.
5. Do not claim that a company succeeded or failed
   unless the provided evidence explicitly says so.
6. Historical failure evidence may be used only for
   failure mechanisms explicitly stated in
   failure_reason.
7. Company analogue evidence may be used only to show
   similarity in business type, category, product,
   location, company size, founding year, or other
   fields explicitly provided.
8. If the evidence is weak, mixed, or insufficient,
   explicitly say so.
9. Do not create exact risk probabilities.
10. Do not recommend conclusions beyond what the
    evidence supports.
11. Every important observation must include evidence
    references using only the IDs supplied below.
12. Return JSON only.

Allowed assessment values:

- "supported"
- "partially_supported"
- "challenged"
- "mixed"
- "insufficient_evidence"

Required JSON schema:

{
  "assessment": "one allowed value",
  "assessment_summary": "short grounded explanation",
  "supporting_signals": [
    {
      "statement": "grounded statement",
      "evidence_refs": ["COMPANY_1"]
    }
  ],
  "risk_signals": [
    {
      "statement": "grounded statement",
      "evidence_refs": ["FAILURE_1"]
    }
  ],
  "failure_mechanisms": [
    {
      "mechanism": "failure mechanism explicitly supported by failure evidence",
      "evidence_refs": ["FAILURE_1"]
    }
  ],
  "evidence_gaps": [
    "what cannot be concluded from available evidence"
  ]
}
""".strip()


    user_prompt = (
        "Evaluate the assumption using ONLY the "
        "following evidence.\n\n"
        + json.dumps(
            evidence_payload,
            indent=2,
            ensure_ascii=False
        )
    )


    raw_content = (
        call_openrouter_cached(
            system_prompt=
                system_prompt,

            user_prompt=
                user_prompt,

            temperature=0.0,
        )
    )


    parsed = (
        parse_llm_json(
            raw_content
        )
    )


    return {
        "assumption_id":
            assumption_packet[
                "assumption_id"
            ],

        "assumption":
            assumption_packet[
                "assumption"
            ],

        "evaluation":
            parsed,

        "raw_response":
            raw_content,
    }

In [55]:
# CELL 51 - REDEFINE GROUNDING REVIEWER WITH CACHE

def review_evaluation_grounding(
    assumption_packet,
    evaluation,
):
    """
    Cached strict second-pass grounding review.
    """

    company_with_refs = []

    for index, company in enumerate(
        assumption_packet[
            "company_analogues"
        ],
        start=1
    ):

        company_with_refs.append(
            {
                "evidence_ref":
                    f"COMPANY_{index}",

                **company,
            }
        )


    failures_with_refs = []

    for index, failure in enumerate(
        assumption_packet[
            "historical_failures"
        ],
        start=1
    ):

        failures_with_refs.append(
            {
                "evidence_ref":
                    f"FAILURE_{index}",

                **failure,
            }
        )


    review_payload = {
        "assumption":
            assumption_packet[
                "assumption"
            ],

        "evidence": {
            "company_analogues":
                company_with_refs,

            "historical_failures":
                failures_with_refs,
        },

        "evaluation_to_review":
            evaluation,
    }


    system_prompt = """
You are a strict evidence-grounding reviewer.

Your ONLY task is to remove or weaken claims that are
not directly supported by the supplied evidence.

Do NOT add any new analysis.

STRICT RULES:

1. Use only the supplied evidence.
2. Do not use outside knowledge.
3. Do not add new facts, companies, mechanisms,
   statistics, probabilities, or explanations.
4. A claim is valid only if the cited evidence directly
   supports the factual content of that claim.
5. Company size, founding year, category, or description
   must NOT be converted into claims about traction,
   success, market share, dominance, adoption, financial
   strength, or product-market fit unless explicitly
   stated in the evidence.
6. Funding amount must NOT be converted into a claim
   about success, demand, traction, or product quality.
7. Historical failure mechanisms must reflect only
   what is explicitly stated in failure_reason.
8. If a claim contains both supported and unsupported
   content, rewrite it to retain only the supported part.
9. If nothing remains supported, remove the claim.
10. Preserve evidence_refs only when they directly
    support the retained claim.
11. Do not make the assessment stronger than the
    evidence allows.
12. When direct evidence is absent, prefer
    "insufficient_evidence".
13. Return JSON only, using exactly the same schema
    as the evaluation being reviewed.
""".strip()


    user_prompt = (
        json.dumps(
            review_payload,
            indent=2,
            ensure_ascii=False
        )
    )


    raw_content = (
        call_openrouter_cached(
            system_prompt=
                system_prompt,

            user_prompt=
                user_prompt,

            temperature=0.0,
        )
    )


    reviewed = (
        parse_llm_json(
            raw_content
        )
    )


    return reviewed

In [56]:
# CELL 52 - BUILD FINAL REVIEWED ASSUMPTION EVALUATIONS

def build_reviewed_assumption_evaluations(
    qualitative_assumptions
):
    """
    Run:
    evidence packet
    -> grounded evaluator
    -> deterministic validation
    -> strict grounding reviewer
    -> deterministic validation again

    Returns only reviewed evaluations.
    """

    final_evaluations = []


    for assumption_item in qualitative_assumptions:

        packet = (
            build_assumption_evidence_packet(
                assumption_item
            )
        )


        # ==================================
        # Initial grounded evaluation
        # ==================================

        initial_result = (
            evaluate_assumption_with_evidence(
                packet
            )
        )


        initial_validation = (
            validate_grounded_evaluation(
                packet,
                initial_result[
                    "evaluation"
                ]
            )
        )


        if not initial_validation[
            "is_valid"
        ]:

            raise ValueError(
                "Initial grounded evaluation "
                f"failed validation for "
                f"{initial_result['assumption_id']}: "
                f"{initial_validation['errors']}"
            )


        # ==================================
        # Strict grounding review
        # ==================================

        reviewed_evaluation = (
            review_evaluation_grounding(
                packet,
                initial_result[
                    "evaluation"
                ]
            )
        )


        reviewed_validation = (
            validate_grounded_evaluation(
                packet,
                reviewed_evaluation
            )
        )


        if not reviewed_validation[
            "is_valid"
        ]:

            raise ValueError(
                "Reviewed evaluation "
                f"failed validation for "
                f"{initial_result['assumption_id']}: "
                f"{reviewed_validation['errors']}"
            )


        final_evaluations.append(
            {
                "assumption_id":
                    initial_result[
                        "assumption_id"
                    ],

                "assumption":
                    initial_result[
                        "assumption"
                    ],

                "evaluation":
                    reviewed_evaluation,

                "grounding_validation": {
                    "is_valid":
                        reviewed_validation[
                            "is_valid"
                        ],

                    "warnings":
                        reviewed_validation[
                            "warnings"
                        ],
                },
            }
        )


    return final_evaluations

In [57]:
# CELL 53 - BUILD FINAL GROUNDED STRESS-TEST RESPONSE

def build_final_stress_test_response(
    analysis_result
):
    """
    Attach reviewed, evidence-grounded
    assumption evaluations to the analysis.
    """

    qualitative = (
        analysis_result[
            "evidence"
        ][
            "qualitative_evidence"
        ]
    )


    reviewed_evaluations = (
        build_reviewed_assumption_evaluations(
            qualitative[
                "assumptions"
            ]
        )
    )


    return {
        "decision":
            analysis_result[
                "decision"
            ],

        "processing_time_seconds":
            analysis_result.get(
                "processing_time_seconds"
            ),

        "summary":
            analysis_result[
                "summary"
            ],

        "assumption_analysis":
            analysis_result[
                "assumption_analysis"
            ],

        "assumption_evaluations":
            reviewed_evaluations,

        "evidence":
            analysis_result[
                "evidence"
            ],
    }

In [58]:
# CELL 54 - TEST FINAL GROUNDED RESPONSE

final_stress_test_response = (
    build_final_stress_test_response(
        full_user_analysis
    )
)


print(
    "FINAL STRESS-TEST RESPONSE READY"
)

print(
    "Decision:",
    final_stress_test_response[
        "decision"
    ]
)

print(
    "Assumptions:",
    len(
        final_stress_test_response[
            "assumption_evaluations"
        ]
    )
)


for item in final_stress_test_response[
    "assumption_evaluations"
]:

    print(
        "\n" + "=" * 100
    )

    print(
        item[
            "assumption_id"
        ],
        "-",
        item[
            "assumption"
        ]
    )

    print(
        "Assessment:",
        item[
            "evaluation"
        ].get(
            "assessment"
        )
    )

    print(
        "Grounding valid:",
        item[
            "grounding_validation"
        ][
            "is_valid"
        ]
    )

LLM CACHE MISS → calling OpenRouter
LLM CACHE MISS → calling OpenRouter
LLM CACHE MISS → calling OpenRouter
LLM CACHE MISS → calling OpenRouter
LLM CACHE MISS → calling OpenRouter
LLM CACHE MISS → calling OpenRouter
LLM CACHE MISS → calling OpenRouter
LLM CACHE MISS → calling OpenRouter
LLM CACHE MISS → calling OpenRouter
LLM CACHE MISS → calling OpenRouter
FINAL STRESS-TEST RESPONSE READY
Decision: Launch an AI-powered SaaS platform for small businesses
that automates repetitive operational workflows.
Assumptions: 5

A1 - Small businesses have sufficient willingness and budget to pay a recurring SaaS subscription for AI-powered workflow automation, rather than relying on free or manual alternatives.
Assessment: insufficient_evidence
Grounding valid: True

A2 - The AI technology can reliably and accurately automate a meaningful share of repetitive operational workflows across diverse small business types without requiring extensive customization per client.
Assessment: insufficient_evi

In [59]:
# CELL 55 - BUILD DETERMINISTIC DECISION SCORECARD

def build_decision_scorecard(
    final_response
):
    """
    Build a deterministic scorecard from reviewed
    assumption evaluations and financial flags.

    No LLM.
    No invented probability.
    No arbitrary numeric risk score.
    """

    evaluations = (
        final_response[
            "assumption_evaluations"
        ]
    )


    assessment_counts = {
        "supported": 0,
        "partially_supported": 0,
        "challenged": 0,
        "mixed": 0,
        "insufficient_evidence": 0,
    }


    for item in evaluations:

        assessment = (
            item[
                "evaluation"
            ]
            .get(
                "assessment"
            )
        )


        if assessment in assessment_counts:

            assessment_counts[
                assessment
            ] += 1


    # ==================================
    # Financial flags
    # ==================================

    financial_evidence = (
        final_response[
            "evidence"
        ]
        .get(
            "financial_evidence"
        )
    )


    financial_flags = []

    if financial_evidence:

        financial_flags = (
            financial_evidence.get(
                "flags",
                []
            )
        )


    high_severity_financial_flags = sum(
        1
        for flag in financial_flags
        if flag.get(
            "severity"
        ) == "high"
    )


    # ==================================
    # Evidence coverage
    # ==================================

    total_assumptions = len(
        evaluations
    )


    assumptions_with_direct_evidence = (
        total_assumptions
        - assessment_counts[
            "insufficient_evidence"
        ]
    )


    evidence_coverage_ratio = (
        assumptions_with_direct_evidence
        / total_assumptions
        if total_assumptions > 0
        else None
    )


    return {
        "assumption_count":
            total_assumptions,

        "assessment_counts":
            assessment_counts,

        "assumptions_with_direct_evidence":
            assumptions_with_direct_evidence,

        "assumptions_with_insufficient_evidence":
            assessment_counts[
                "insufficient_evidence"
            ],

        "evidence_coverage_ratio":
            evidence_coverage_ratio,

        "financial_flag_count":
            len(
                financial_flags
            ),

        "high_severity_financial_flag_count":
            high_severity_financial_flags,

        "has_negative_financial_signal":
            high_severity_financial_flags > 0,
    }

In [60]:
# CELL 56 - COLLECT GROUNDED RISKS AND EVIDENCE GAPS

def collect_grounded_risk_summary(
    final_response
):
    """
    Collect only reviewed and grounded:
    - failure mechanisms
    - risk signals
    - evidence gaps

    No new interpretation is introduced.
    """

    failure_mechanisms = []
    risk_signals = []
    evidence_gaps = []


    for assumption_item in final_response[
        "assumption_evaluations"
    ]:

        assumption_id = (
            assumption_item[
                "assumption_id"
            ]
        )

        assumption_text = (
            assumption_item[
                "assumption"
            ]
        )

        evaluation = (
            assumption_item[
                "evaluation"
            ]
        )


        # ==================================
        # Failure mechanisms
        # ==================================

        for mechanism in evaluation.get(
            "failure_mechanisms",
            []
        ):

            failure_mechanisms.append(
                {
                    "assumption_id":
                        assumption_id,

                    "assumption":
                        assumption_text,

                    "mechanism":
                        mechanism.get(
                            "mechanism"
                        ),

                    "evidence_refs":
                        mechanism.get(
                            "evidence_refs",
                            []
                        ),
                }
            )


        # ==================================
        # Risk signals
        # ==================================

        for risk in evaluation.get(
            "risk_signals",
            []
        ):

            risk_signals.append(
                {
                    "assumption_id":
                        assumption_id,

                    "assumption":
                        assumption_text,

                    "statement":
                        risk.get(
                            "statement"
                        ),

                    "evidence_refs":
                        risk.get(
                            "evidence_refs",
                            []
                        ),
                }
            )


        # ==================================
        # Evidence gaps
        # ==================================

        for gap in evaluation.get(
            "evidence_gaps",
            []
        ):

            gap = str(
                gap
            ).strip()


            if gap:

                evidence_gaps.append(
                    {
                        "assumption_id":
                            assumption_id,

                        "assumption":
                            assumption_text,

                        "gap":
                            gap,
                    }
                )


    return {
        "failure_mechanisms":
            failure_mechanisms,

        "risk_signals":
            risk_signals,

        "evidence_gaps":
            evidence_gaps,
    }

In [61]:
# CELL 57 - BUILD DETERMINISTIC VALIDATION PRIORITIES

def build_validation_priorities(
    final_response
):
    """
    Build validation priorities directly from:
    - insufficient-evidence assumptions
    - explicit evidence gaps
    - deterministic financial flags

    No LLM.
    """

    priorities = []

    priority_number = 1


    # ==================================
    # Assumption evidence gaps
    # ==================================

    for item in final_response[
        "assumption_evaluations"
    ]:

        evaluation = (
            item[
                "evaluation"
            ]
        )


        assessment = (
            evaluation.get(
                "assessment"
            )
        )


        gaps = (
            evaluation.get(
                "evidence_gaps",
                []
            )
        )


        if (
            assessment
            == "insufficient_evidence"
        ):

            priorities.append(
                {
                    "priority":
                        priority_number,

                    "type":
                        "assumption_validation",

                    "assumption_id":
                        item[
                            "assumption_id"
                        ],

                    "assumption":
                        item[
                            "assumption"
                        ],

                    "reason":
                        (
                            "Available evidence is "
                            "insufficient to directly "
                            "evaluate this assumption."
                        ),

                    "evidence_gaps":
                        gaps,
                }
            )

            priority_number += 1


    # ==================================
    # Financial flags
    # ==================================

    financial_evidence = (
        final_response[
            "evidence"
        ]
        .get(
            "financial_evidence"
        )
    )


    if financial_evidence:

        for flag in financial_evidence.get(
            "flags",
            []
        ):

            priorities.append(
                {
                    "priority":
                        priority_number,

                    "type":
                        "financial_validation",

                    "flag_code":
                        flag.get(
                            "code"
                        ),

                    "severity":
                        flag.get(
                            "severity"
                        ),

                    "reason":
                        flag.get(
                            "message"
                        ),
                }
            )

            priority_number += 1


    return priorities

In [62]:
# CELL 58 - TEST SCORECARD AND VALIDATION PRIORITIES

decision_scorecard = (
    build_decision_scorecard(
        final_stress_test_response
    )
)


grounded_risk_summary = (
    collect_grounded_risk_summary(
        final_stress_test_response
    )
)


validation_priorities = (
    build_validation_priorities(
        final_stress_test_response
    )
)


print(
    "DECISION SCORECARD"
)

print(
    json.dumps(
        decision_scorecard,
        indent=2,
        ensure_ascii=False
    )
)


print(
    "\nGROUNDED RISK SUMMARY"
)

print(
    json.dumps(
        grounded_risk_summary,
        indent=2,
        ensure_ascii=False
    )
)


print(
    "\nVALIDATION PRIORITIES"
)

print(
    json.dumps(
        validation_priorities,
        indent=2,
        ensure_ascii=False
    )
)

DECISION SCORECARD
{
  "assumption_count": 5,
  "assessment_counts": {
    "supported": 0,
    "partially_supported": 0,
    "challenged": 0,
    "mixed": 0,
    "insufficient_evidence": 5
  },
  "assumptions_with_direct_evidence": 0,
  "assumptions_with_insufficient_evidence": 5,
  "evidence_coverage_ratio": 0.0,
  "financial_flag_count": 1,
  "high_severity_financial_flag_count": 1,
  "has_negative_financial_signal": true
}

GROUNDED RISK SUMMARY
{
  "failure_mechanisms": [
    {
      "assumption_id": "A1",
      "assumption": "Small businesses have sufficient willingness and budget to pay a recurring SaaS subscription for AI-powered workflow automation, rather than relying on free or manual alternatives.",
      "mechanism": "Price sensitivity leading to slow user adoption when products carry a cost barrier",
      "evidence_refs": [
        "FAILURE_2"
      ]
    },
    {
      "assumption_id": "A2",
      "assumption": "The AI technology can reliably and accurately automate a me

In [63]:
# CELL 59 - BUILD STRICT FINAL GROUNDED RISK SUMMARY

def build_strict_grounded_risk_summary(
    final_response
):
    """
    Build the final risk summary with stricter rules.

    Rules:
    - FAILURE_* evidence may support risks / failure mechanisms.
    - COMPANY_* evidence is analogue/context only.
    - Company analogue evidence is NOT used to infer risk.
    - No new interpretation is introduced.
    """

    failure_mechanisms = []
    historical_risk_signals = []
    company_analogue_context = []
    evidence_gaps = []


    for assumption_item in final_response[
        "assumption_evaluations"
    ]:

        assumption_id = (
            assumption_item[
                "assumption_id"
            ]
        )

        assumption_text = (
            assumption_item[
                "assumption"
            ]
        )

        evaluation = (
            assumption_item[
                "evaluation"
            ]
        )


        # ==================================
        # Historical failure mechanisms only
        # ==================================

        for mechanism in evaluation.get(
            "failure_mechanisms",
            []
        ):

            refs = mechanism.get(
                "evidence_refs",
                []
            )


            failure_refs = [
                ref
                for ref in refs
                if str(ref).startswith(
                    "FAILURE_"
                )
            ]


            if not failure_refs:
                continue


            failure_mechanisms.append(
                {
                    "assumption_id":
                        assumption_id,

                    "assumption":
                        assumption_text,

                    "mechanism":
                        mechanism.get(
                            "mechanism"
                        ),

                    "evidence_refs":
                        failure_refs,
                }
            )


        # ==================================
        # Historical risk signals only
        # ==================================

        for risk in evaluation.get(
            "risk_signals",
            []
        ):

            refs = risk.get(
                "evidence_refs",
                []
            )


            failure_refs = [
                ref
                for ref in refs
                if str(ref).startswith(
                    "FAILURE_"
                )
            ]


            if failure_refs:

                historical_risk_signals.append(
                    {
                        "assumption_id":
                            assumption_id,

                        "assumption":
                            assumption_text,

                        "statement":
                            risk.get(
                                "statement"
                            ),

                        "evidence_refs":
                            failure_refs,
                    }
                )


        # ==================================
        # Company analogue = context only
        # ==================================

        qualitative_item = next(
            (
                item
                for item in final_response[
                    "evidence"
                ][
                    "qualitative_evidence"
                ][
                    "assumptions"
                ]
                if item[
                    "assumption_id"
                ] == assumption_id
            ),
            None
        )


        if qualitative_item:

            for company in qualitative_item.get(
                "company_analogues",
                []
            ):

                company_analogue_context.append(
                    {
                        "assumption_id":
                            assumption_id,

                        "rank":
                            company.get(
                                "rank"
                            ),

                        "name":
                            company.get(
                                "name"
                            ),

                        "semantic_score":
                            company.get(
                                "semantic_score"
                            ),

                        "short_description":
                            company.get(
                                "short_description"
                            ),

                        "categories":
                            company.get(
                                "categories"
                            ),

                        "employee_range":
                            company.get(
                                "employee_range"
                            ),

                        "founded_year":
                            company.get(
                                "founded_year"
                            ),
                    }
                )


        # ==================================
        # Evidence gaps
        # ==================================

        for gap in evaluation.get(
            "evidence_gaps",
            []
        ):

            gap = str(
                gap
            ).strip()


            if gap:

                evidence_gaps.append(
                    {
                        "assumption_id":
                            assumption_id,

                        "assumption":
                            assumption_text,

                        "gap":
                            gap,
                    }
                )


    return {
        "historical_failure_mechanisms":
            failure_mechanisms,

        "historical_risk_signals":
            historical_risk_signals,

        "company_analogue_context":
            company_analogue_context,

        "evidence_gaps":
            evidence_gaps,
    }

In [64]:
# CELL 60 - BUILD FINAL COMPACT DECISION RESPONSE

def build_compact_decision_response(
    final_response
):
    """
    Build final compact API-friendly response.

    No new LLM calls.
    """

    scorecard = (
        build_decision_scorecard(
            final_response
        )
    )


    risk_summary = (
        build_strict_grounded_risk_summary(
            final_response
        )
    )


    validation_priorities = (
        build_validation_priorities(
            final_response
        )
    )


    financial_evidence = (
        final_response[
            "evidence"
        ].get(
            "financial_evidence"
        )
    )


    assumption_results = []


    for item in final_response[
        "assumption_evaluations"
    ]:

        evaluation = (
            item[
                "evaluation"
            ]
        )


        assumption_results.append(
            {
                "assumption_id":
                    item[
                        "assumption_id"
                    ],

                "assumption":
                    item[
                        "assumption"
                    ],

                "assessment":
                    evaluation.get(
                        "assessment"
                    ),

                "assessment_summary":
                    evaluation.get(
                        "assessment_summary"
                    ),

                "evidence_gaps":
                    evaluation.get(
                        "evidence_gaps",
                        []
                    ),

                "grounding_valid":
                    item[
                        "grounding_validation"
                    ][
                        "is_valid"
                    ],
            }
        )


    return {
        "decision":
            final_response[
                "decision"
            ],

        "scorecard":
            scorecard,

        "assumption_results":
            assumption_results,

        "risk_evidence":
            risk_summary,

        "financial_evidence":
            financial_evidence,

        "validation_priorities":
            validation_priorities,

        "source_metadata": {
            "company_candidate_universe":
                final_response[
                    "summary"
                ][
                    "company_candidate_universe_size"
                ],

            "historical_failure_cases":
                final_response[
                    "summary"
                ][
                    "historical_failure_universe_size"
                ],

            "idx_benchmark_ratios":
                len(
                    idx_benchmark_overall_df
                ),
        },
    }

In [65]:
# CELL 61 - TEST FINAL COMPACT DECISION RESPONSE

compact_decision_response = (
    build_compact_decision_response(
        final_stress_test_response
    )
)


print(
    "FINAL COMPACT RESPONSE"
)


print(
    json.dumps(
        compact_decision_response,
        indent=2,
        ensure_ascii=False
    )[:30000]
)

FINAL COMPACT RESPONSE
{
  "decision": "Launch an AI-powered SaaS platform for small businesses\nthat automates repetitive operational workflows.",
  "scorecard": {
    "assumption_count": 5,
    "assessment_counts": {
      "supported": 0,
      "partially_supported": 0,
      "challenged": 0,
      "mixed": 0,
      "insufficient_evidence": 5
    },
    "assumptions_with_direct_evidence": 0,
    "assumptions_with_insufficient_evidence": 5,
    "evidence_coverage_ratio": 0.0,
    "financial_flag_count": 1,
    "high_severity_financial_flag_count": 1,
    "has_negative_financial_signal": true
  },
  "assumption_results": [
    {
      "assumption_id": "A1",
      "assumption": "Small businesses have sufficient willingness and budget to pay a recurring SaaS subscription for AI-powered workflow automation, rather than relying on free or manual alternatives.",
      "assessment": "insufficient_evidence",
      "assessment_summary": "The provided evidence does not directly address whether 

In [66]:
# CELL 62 - BUILD SAFER FINAL RISK EVIDENCE

def build_final_risk_evidence(
    final_response
):
    """
    Final conservative risk evidence.

    Historical risks are represented ONLY by
    grounded failure mechanisms.

    Company analogues remain context only.
    Evidence gaps remain separate.
    """

    strict_summary = (
        build_strict_grounded_risk_summary(
            final_response
        )
    )


    return {
        "historical_failure_mechanisms":
            strict_summary[
                "historical_failure_mechanisms"
            ],

        "company_analogue_context":
            strict_summary[
                "company_analogue_context"
            ],

        "evidence_gaps":
            strict_summary[
                "evidence_gaps"
            ],
    }

In [71]:
# CELL 63 - BUILD DETERMINISTIC VALIDATION EXPERIMENTS
# REVISED WITH MORE SPECIFIC CLASSIFICATION RULES

def recommend_validation_experiment(
    assumption_id,
    assumption,
):
    """
    Deterministic experiment recommendation.

    Recommendations are NOT evidence.
    No invented probability or threshold.
    """

    text = str(
        assumption
    ).lower()


    # ==================================
    # CAC / LTV economics
    # Most specific financial-growth rule
    # ==================================

    if any(
        keyword in text
        for keyword in [
            "customer acquisition cost",
            "acquisition costs",
            "acquisition cost",
            "cac",
            "lifetime value",
            "ltv",
        ]
    ):

        return {
            "assumption_id":
                assumption_id,

            "type":
                "recommended_experiment",

            "experiment":
                "Small-scale acquisition economics test",

            "method":
                (
                    "Run a limited customer acquisition "
                    "test using the intended channel and "
                    "track spend through qualified lead, "
                    "conversion, and initial retention."
                ),

            "measure":
                (
                    "Calculate observed acquisition cost "
                    "from actual spend and compare it with "
                    "revenue and retention data collected "
                    "from the same test."
                ),

            "note":
                (
                    "Do not extrapolate lifetime value "
                    "until sufficient retention evidence exists."
                ),
        }


    # ==================================
    # Competition
    # Require explicitly competitive concepts.
    # Do NOT classify from 'alternatives' alone.
    # ==================================

    competition_keywords = [
        "competitive landscape",
        "competition",
        "competitor",
        "competitors",
        "incumbent",
        "incumbents",
        "dominant",
        "well-funded startups",
        "well funded startups",
        "comparable ai",
        "market saturation",
        "competitive intensity",
    ]


    if any(
        keyword in text
        for keyword in competition_keywords
    ):

        return {
            "assumption_id":
                assumption_id,

            "type":
                "recommended_experiment",

            "experiment":
                "Competitive alternative audit",

            "method":
                (
                    "Identify the alternatives currently "
                    "used by target customers and compare "
                    "their product scope, positioning, "
                    "available pricing information, and "
                    "workflow coverage."
                ),

            "measure":
                (
                    "Record which alternatives customers "
                    "already use, what comparable products "
                    "are available, and why target users "
                    "would or would not switch."
                ),

            "note":
                (
                    "Company analogue similarity alone "
                    "does not establish market dominance, "
                    "customer adoption, or competitive strength."
                ),
        }


    # ==================================
    # CAC / LTV already handled above.
    # Digital readiness / integration
    # ==================================

    if any(
        keyword in text
        for keyword in [
            "digital maturity",
            "structured operational data",
            "integration readiness",
            "digital readiness",
            "onboard",
            "cloud-based",
            "cloud based",
        ]
    ):

        return {
            "assumption_id":
                assumption_id,

            "type":
                "recommended_experiment",

            "experiment":
                "Customer integration-readiness audit",

            "method":
                (
                    "Interview or onboard a small set of "
                    "representative target businesses and "
                    "inspect the systems, data formats, "
                    "and integrations required."
                ),

            "measure":
                (
                    "Record missing data, manual setup, "
                    "integration blockers, and onboarding "
                    "effort observed."
                ),

            "note":
                (
                    "Do not infer readiness from company "
                    "size alone."
                ),
        }


    # ==================================
    # AI reliability / capability
    # ==================================

    if any(
        keyword in text
        for keyword in [
            "reliably",
            "accurately",
            "accuracy",
            "customization",
            "generalization",
            "technical capability",
            "meaningful share",
        ]
    ):

        return {
            "assumption_id":
                assumption_id,

            "type":
                "recommended_experiment",

            "experiment":
                "Workflow automation pilot",

            "method":
                (
                    "Run the proposed AI workflow on a "
                    "representative set of real target "
                    "workflows from several intended "
                    "customer types."
                ),

            "measure":
                (
                    "Track task completion quality, "
                    "manual corrections, failures, and "
                    "customization required per workflow."
                ),

            "note":
                (
                    "Define acceptable performance "
                    "criteria before the pilot."
                ),
        }


    # ==================================
    # Willingness to pay / pricing
    # ==================================

    if any(
        keyword in text
        for keyword in [
            "willingness",
            "willing to pay",
            "budget",
            "subscription",
            "recurring saas",
            "price",
            "pricing",
            "pay a recurring",
        ]
    ):

        return {
            "assumption_id":
                assumption_id,

            "type":
                "recommended_experiment",

            "experiment":
                "Willingness-to-pay validation",

            "method":
                (
                    "Present the proposed product and "
                    "pricing to representative target "
                    "customers and collect explicit "
                    "purchase, preorder, paid-pilot, or "
                    "rejection signals."
                ),

            "measure":
                (
                    "Record actual commitment signals "
                    "and stated reasons for acceptance "
                    "or rejection."
                ),

            "note":
                (
                    "Define the success criterion before "
                    "running the experiment."
                ),
        }


    # ==================================
    # Generic fallback
    # ==================================

    return {
        "assumption_id":
            assumption_id,

        "type":
            "recommended_experiment",

        "experiment":
            "Direct assumption validation",

        "method":
            (
                "Design the smallest test that exposes "
                "the assumption to representative target "
                "users or operating conditions."
            ),

        "measure":
            (
                "Collect direct observable evidence that "
                "would support or challenge the assumption."
            ),

        "note":
            (
                "Define the success criterion before "
                "running the test."
            ),
    }


def build_validation_experiments(
    final_response
):
    experiments = []


    for item in final_response[
        "assumption_evaluations"
    ]:

        if (
            item[
                "evaluation"
            ].get(
                "assessment"
            )
            == "insufficient_evidence"
        ):

            experiments.append(
                recommend_validation_experiment(
                    item[
                        "assumption_id"
                    ],
                    item[
                        "assumption"
                    ],
                )
            )


    return experiments

In [72]:
# CELL 64 - TEST FINAL RISK EVIDENCE AND VALIDATION EXPERIMENTS

final_risk_evidence = (
    build_final_risk_evidence(
        final_stress_test_response
    )
)


validation_experiments = (
    build_validation_experiments(
        final_stress_test_response
    )
)


print(
    "FINAL RISK EVIDENCE"
)

print(
    json.dumps(
        final_risk_evidence,
        indent=2,
        ensure_ascii=False
    )
)


print(
    "\nVALIDATION EXPERIMENTS"
)

print(
    json.dumps(
        validation_experiments,
        indent=2,
        ensure_ascii=False
    )
)

FINAL RISK EVIDENCE
{
  "historical_failure_mechanisms": [
    {
      "assumption_id": "A1",
      "assumption": "Small businesses have sufficient willingness and budget to pay a recurring SaaS subscription for AI-powered workflow automation, rather than relying on free or manual alternatives.",
      "mechanism": "Price sensitivity leading to slow user adoption when products carry a cost barrier",
      "evidence_refs": [
        "FAILURE_2"
      ]
    },
    {
      "assumption_id": "A2",
      "assumption": "The AI technology can reliably and accurately automate a meaningful share of repetitive operational workflows across diverse small business types without requiring extensive customization per client.",
      "mechanism": "AI technology not being mature enough to deliver on its automation promises (Kite: 'the tech is not ready yet')",
      "evidence_refs": [
        "FAILURE_5"
      ]
    },
    {
      "assumption_id": "A2",
      "assumption": "The AI technology can reliabl

In [73]:
# CELL 65 - BUILD FINAL API RESPONSE

def build_final_api_response(
    final_response
):
    """
    Build the final backend-ready response.

    No new LLM calls.
    """

    return {
        "decision":
            final_response[
                "decision"
            ],

        "scorecard":
            build_decision_scorecard(
                final_response
            ),

        "assumptions": [
            {
                "assumption_id":
                    item[
                        "assumption_id"
                    ],

                "assumption":
                    item[
                        "assumption"
                    ],

                "assessment":
                    item[
                        "evaluation"
                    ].get(
                        "assessment"
                    ),

                "assessment_summary":
                    item[
                        "evaluation"
                    ].get(
                        "assessment_summary"
                    ),

                "evidence_gaps":
                    item[
                        "evaluation"
                    ].get(
                        "evidence_gaps",
                        []
                    ),

                "grounding_valid":
                    item[
                        "grounding_validation"
                    ][
                        "is_valid"
                    ],
            }

            for item in final_response[
                "assumption_evaluations"
            ]
        ],

        "risk_evidence":
            build_final_risk_evidence(
                final_response
            ),

        "financial_evidence":
            final_response[
                "evidence"
            ].get(
                "financial_evidence"
            ),

        "validation_experiments":
            build_validation_experiments(
                final_response
            ),

        "source_metadata": {
            "company_candidate_universe":
                final_response[
                    "summary"
                ][
                    "company_candidate_universe_size"
                ],

            "historical_failure_cases":
                final_response[
                    "summary"
                ][
                    "historical_failure_universe_size"
                ],

            "idx_benchmark_ratios":
                len(
                    idx_benchmark_overall_df
                ),
        },
    }

In [74]:
# CELL 66 - TEST FINAL API RESPONSE

final_api_response = (
    build_final_api_response(
        final_stress_test_response
    )
)


print(
    "FINAL API RESPONSE READY"
)

print(
    "Decision:",
    final_api_response[
        "decision"
    ]
)

print(
    "Assumptions:",
    len(
        final_api_response[
            "assumptions"
        ]
    )
)

print(
    "Validation experiments:",
    len(
        final_api_response[
            "validation_experiments"
        ]
    )
)

print(
    "Historical failure mechanisms:",
    len(
        final_api_response[
            "risk_evidence"
        ][
            "historical_failure_mechanisms"
        ]
    )
)

FINAL API RESPONSE READY
Decision: Launch an AI-powered SaaS platform for small businesses
that automates repetitive operational workflows.
Assumptions: 5
Validation experiments: 5
Historical failure mechanisms: 3


In [75]:
# CELL 67 - SAVE FINAL API RESPONSE EXAMPLE

FINAL_RESPONSE_EXAMPLE_FILE = Path(
    "data/outputs/final_analysis_response_example.json"
)


FINAL_RESPONSE_EXAMPLE_FILE.parent.mkdir(parents=True, exist_ok=True)

with open(
    FINAL_RESPONSE_EXAMPLE_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        final_api_response,
        file,
        indent=2,
        ensure_ascii=False
    )


print(
    "Saved:",
    FINAL_RESPONSE_EXAMPLE_FILE
)

print(
    "File exists:",
    FINAL_RESPONSE_EXAMPLE_FILE.exists()
)

print(
    "Size KB:",
    round(
        FINAL_RESPONSE_EXAMPLE_FILE.stat().st_size
        / 1024,
        2
    )
)

Saved: data\final_analysis_response_example.json
File exists: True
Size KB: 33.84
